# Laboratori 9 — Max-Flow / Min-Cut & Ford-Fulkerson vs Edmonds-Karp

---

### Çfarë është Max-Flow?
Imagjino një sistem tubacioni (ose rrjet kompjuterik). Rrjeti ka:
- Një **burim (source S)** — nga ku rrjedh uji/të dhënat
- Një **kulm (sink T)** — ku arrin uji/të dhënat
- **Brinjë me kapacitete** — sa maximum mund të kalojë nëpër çdo tub

**Problemi:** Cila është sasia maksimale që mund të rrjedhë nga S te T?

---

### Termat kyç

| Term | Kuptimi |
|------|---------|
| **Kapaciteti c(u,v)** | Sasia maksimale që mund të kalojë nëpër brinjën u→v |
| **Rrjedha f(u,v)** | Sasia reale që kalon nëpër brinjën u→v |
| **Graf mbetës** | Grafi i përditësuar që tregon kapacitetin e mbetur pas çdo iterimi |
| **Rrugë rritëse (augmenting path)** | Rrugë nga S te T në graf-in mbetës me kapacitet > 0 |
| **Prerje (cut)** | Ndarje e kulmeve në dy grupe S-side dhe T-side |
| **Kapaciteti i prerjes** | Shuma e kapaciteteve të brinjëve që kalojnë nga S-side → T-side |
| **Ngushtica (bottleneck)** | Brinja me kapacitetin mbetës minimal në rrugën rritëse |

---

### Kushtet e Rrjedhës

1. **Kufizimi i kapacitetit:** `0 ≤ f(u,v) ≤ c(u,v)` — rrjedha nuk mund të kalojë kapacitetin
2. **Ruajtja e rrjedhës:** Për çdo kulm (përveç S dhe T): `Σ f(u,v) in = Σ f(v,w) out`
3. **Anti-simetria:** `f(u,v) = -f(v,u)` — rrjedha në drejtim të kundërt është negative

---

### Algoritmet kryesore

| Algoritmi | Strategjia kërkimit | Kompleksiteti | Garanton min rrugë? |
|-----------|--------------------:|:-------------:|:-------------------:|
| **Ford-Fulkerson** | DFS (Depth-First Search) | O(E · f) | ❌ |
| **Edmonds-Karp** | BFS (Breadth-First Search) | O(V · E²) | ✅ |

> **Shënim:** Edmonds-Karp është variant i Ford-Fulkerson që përdor BFS.

---

### Teorema Themelore: Max-Flow = Min-Cut

> **"Rrjedha maksimale nga S te T është gjithmonë e barabartë me kapacitetin e prerjes minimale."**

Kjo do të thotë:
- Nëse gjejmë rrjedhën maksimale, kemi gjetur automatikisht edhe prerjen minimale
- Prerja minimale identifikon **vendet kritike (bottlenecks)** të rrjetit
- Nëse heqim brinjët e prerjes minimale, S dhe T ndahen plotësisht

---


---

## Ushtrimi 1 — Vizualizim Interaktiv: Max-Flow/Min-Cut

### Konceptet para ushtrimit

**Ky ushtrim ilustron Max-Flow/Min-Cut me një sistem tubacioni uji.**

#### Çfarë do të bësh:
- Do të **presësh tuba** (brinjë) në mënyrë interaktive duke klikuar mbi to
- Synimi: gjej grupin **minimal** të tubave që, nëse priten, ndajnë hidrantin nga depozita
- Kur kapaciteti i prerjes sate = Max-Flow → ke gjetur **Min-Cut**!

#### Si funksionon vizualizimi:
- **Nyjet blu:** janë në anën e burimit (S-side)
- **Nyjet kuq:** janë në anën e kulmit (T-side)
- **Tubat e prerë:** shfaqen me ×
- **Statusi** poshtë tregon nëse prerja jote është e vlefshme

#### Çfarë duhet të vëresh:
- Jo çdo prerje e vlefshme është prerja minimale
- Duhet të gjesh prerjen me **kapacitetin më të vogël**
- Kapaciteti i prerjes minimale = Rrjedha maksimale (Teorema)

> **Ndërface:** Klik tub → e pret/restauron | `M` → tregon min-cut teorik | `R` → reset | `H` → ndihmë


In [2]:
import pygame, math
from collections import defaultdict, deque

pygame.init()

W0, H0 = 1440, 860
SOURCE, SINK = 0, 5
EDGES = [(0,1,10),(0,2,8),(1,3,6),(1,4,4),(2,4,7),(3,5,9),(4,5,8)]
NAMES = {0:"Hidranti",1:"B-1",2:"B-2",3:"B-3",4:"B-4",5:"Depozita"}

C = {
    "bg":(8,14,28), "graph":(10,18,35), "panel":(15,24,46),
    "white":(230,245,255), "dim":(125,148,190), "blue":(70,190,255),
    "cyan":(40,220,255), "yellow":(240,190,40), "red":(255,80,55),
    "green":(60,220,120), "orange":(255,160,60), "pipe":(75,130,165),
    "pipe_dark":(20,35,55), "pipe_hover":(130,200,250), "water":(60,195,230),
    "s":(40,200,255), "t":(255,130,100), "neutral":(110,130,165),
    "btn":(22,34,60), "btn_h":(32,48,80)
}
FONTS = {}


def f(size, bold=False):
    key = (size, bold)
    if key not in FONTS:
        FONTS[key] = pygame.font.SysFont("Segoe UI", size, bold=bold)
    return FONTS[key]


def text(s, msg, size, color, pos, center=False, bold=False):
    img = f(size, bold).render(str(msg), True, color)
    r = img.get_rect(center=pos) if center else img.get_rect(topleft=pos)
    s.blit(img, r)
    return r


def glow(s, x, y, color, radius):
    for r in range(radius, 0, -8):
        surf = pygame.Surface((2*r, 2*r), pygame.SRCALPHA)
        pygame.draw.circle(surf, (*color, int(70*r/radius)), (r, r), r)
        s.blit(surf, (x-r, y-r))


def graph_data():
    cap = defaultdict(lambda: defaultdict(int)); adj = defaultdict(set)
    for u, v, c in EDGES:
        cap[u][v] += c; adj[u].add(v); adj[v].add(u)
    return cap, adj


def max_flow():
    cap, adj = graph_data(); flow = defaultdict(lambda: defaultdict(int))
    while True:
        parent, q = {SOURCE: None}, deque([SOURCE])
        while q and SINK not in parent:
            u = q.popleft()
            for v in sorted(adj[u]):
                if cap[u][v] - flow[u][v] > 0 and v not in parent:
                    parent[v] = u; q.append(v)
        if SINK not in parent: break
        path, cur = [], SINK
        while cur != SOURCE:
            prev = parent[cur]; path.append((prev, cur)); cur = prev
        b = min(cap[u][v] - flow[u][v] for u, v in path)
        for u, v in path:
            flow[u][v] += b; flow[v][u] -= b
    return sum(flow[SOURCE][v] for v in adj[SOURCE]), {u: dict(flow[u]) for u in flow}


def partition(cut):
    adj = defaultdict(set)
    for u, v, _ in EDGES:
        if (u, v) not in cut: adj[u].add(v)
    seen, q = {SOURCE}, deque([SOURCE])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in seen: seen.add(v); q.append(v)
    return seen, set(NAMES) - seen


def min_cut():
    mf, flow = max_flow(); cap, adj = graph_data()
    seen, q = {SOURCE}, deque([SOURCE])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if cap[u][v] - flow.get(u, {}).get(v, 0) > 0 and v not in seen:
                seen.add(v); q.append(v)
    return [(u, v) for u, v, _ in EDGES if u in seen and v not in seen], mf, seen


def dist_point_line(px, py, x1, y1, x2, y2):
    dx, dy = x2-x1, y2-y1; l2 = dx*dx + dy*dy
    if not l2: return math.hypot(px-x1, py-y1)
    t = max(0, min(1, ((px-x1)*dx + (py-y1)*dy) / l2))
    return math.hypot(px-(x1+t*dx), py-(y1+t*dy))


def node_positions(gw, h):
    top, bot = 100, h-80; span = bot-top
    return {
        0:(int(gw*.08), int(top+span*.50)), 1:(int(gw*.32), int(top+span*.22)),
        2:(int(gw*.32), int(top+span*.78)), 3:(int(gw*.62), int(top+span*.22)),
        4:(int(gw*.62), int(top+span*.78)), 5:(int(gw*.88), int(top+span*.50)),
    }


def draw_hydrant(s, x, y, col, tick):
    glow(s, x, y, col, 75)
    pygame.draw.rect(s,(180,40,30),(x-28,y-30,56,60),border_radius=6)
    pygame.draw.rect(s,(235,90,80),(x-28,y-30,56,60),2,border_radius=6)
    pygame.draw.rect(s,(140,30,20),(x-22,y-38,44,12),border_radius=4)
    pygame.draw.rect(s,(220,70,60),(x-22,y-38,44,12),2,border_radius=4)
    pygame.draw.rect(s,(90,95,105),(x-36,y-6,12,18)); pygame.draw.rect(s,(90,95,105),(x+24,y-6,12,18))
    pygame.draw.circle(s,(50,55,65),(x,y),8); pygame.draw.circle(s,(200,210,225),(x,y),8,2)
    a0 = tick*.04
    for i in range(4):
        a = a0+i*math.pi/2
        pygame.draw.line(s,(200,210,225),(x,y),(x+int(math.cos(a)*9), y+int(math.sin(a)*9)),2)
    text(s,"HIDRANTI",13,col,(x,y+48),True,True); text(s,"BURIMI",10,C["dim"],(x,y+63),True)


def draw_junction(s, x, y, name, col, water, tick):
    glow(s, x, y, col, 50)
    pygame.draw.circle(s,(45,60,80),(x,y),30); pygame.draw.circle(s,(140,170,200),(x,y),30,3)
    for a,b in [((x-45,y),(x+45,y)), ((x,y-45),(x,y+45))]:
        pygame.draw.line(s,C["pipe_dark"],a,b,18); pygame.draw.line(s,C["pipe"],a,b,12); pygame.draw.line(s,(200,230,255),a,b,2)
    pygame.draw.circle(s,(35,50,70),(x,y),22); pygame.draw.circle(s,col,(x,y),22,3)
    if water > .5:
        for i in range(4):
            a = tick*.05+i*math.pi/2
            px, py = x+int(math.cos(a)*13), y+int(math.sin(a)*13)
            pygame.draw.circle(s,C["water"],(px,py),4); pygame.draw.circle(s,(200,240,255),(px,py),2)
    text(s,name,15,C["white"],(x,y+50),True,True); text(s,"Brryl tubi",10,C["white"],(x,y+64),True)


def draw_deposit(s, x, y, col, level, tick):
    glow(s, x, y, col, 75)
    pygame.draw.ellipse(s,(70,85,105),(x-42,y-38,84,20)); pygame.draw.rect(s,(50,65,88),(x-42,y-28,84,58))
    pygame.draw.ellipse(s,(45,58,78),(x-42,y+20,84,20))
    pygame.draw.ellipse(s,(160,190,220),(x-42,y-38,84,20),2)
    pygame.draw.line(s,(160,190,220),(x-42,y-28),(x-42,y+30),2); pygame.draw.line(s,(160,190,220),(x+42,y-28),(x+42,y+30),2)
    pygame.draw.ellipse(s,(160,190,220),(x-42,y+20,84,20),2)
    h = int(50*level)
    if h > 2:
        wy = y+24-h
        pygame.draw.rect(s,C["water"],(x-36,wy,72,h)); pygame.draw.ellipse(s,C["water"],(x-36,wy-6,72,14))
        for i in range(4):
            wx = x-28+i*18; off = int(math.sin(tick*.08+i*1.2)*2)
            pygame.draw.line(s,(180,240,255),(wx,wy+off),(wx+14,wy-off),2)
    text(s,"DEPOZITA",13,col,(x,y+52),True,True); text(s,"KULMI",10,C["dim"],(x,y+66),True)


def draw_node(s, n, x, y, col, water, tick):
    if n == SOURCE: draw_hydrant(s, x, y, col, tick)
    elif n == SINK: draw_deposit(s, x, y, col, water, tick)
    else: draw_junction(s, x, y, NAMES[n], col, water, tick)


def draw_pipe(s, x1, y1, x2, y2, cap, cut, hover, water, tick):
    dx, dy = x2-x1, y2-y1; length = math.hypot(dx,dy) or 1; w = max(10, 7+int(cap*.6))
    pygame.draw.line(s,C["pipe_dark"],(x1,y1),(x2,y2),w+8)
    if cut:
        mx,my = (x1+x2)/2, (y1+y2)/2; ux,uy = dx/length, dy/length; gap = 18
        a, b = (mx-ux*gap/2, my-uy*gap/2), (mx+ux*gap/2, my+uy*gap/2)
        pygame.draw.line(s,(60,25,22),(x1,y1),(x2,y2),w)
        pygame.draw.line(s,(130,50,45),(x1,y1),a,w); pygame.draw.line(s,(130,50,45),b,(x2,y2),w)
        pygame.draw.line(s,C["red"],(mx-14,my-14),(mx+14,my+14),4); pygame.draw.line(s,C["red"],(mx-14,my+14),(mx+14,my-14),4)
        for i in range(3):
            t = (tick*.05+i*.3)%1
            pygame.draw.circle(s,C["water"],(int(a[0]+math.sin(tick*.1+i)*3), int(a[1]+8+t*20)),2)
        return
    color = C["pipe_hover"] if hover else C["pipe"]
    pygame.draw.line(s,color,(x1,y1),(x2,y2),w); pygame.draw.line(s,(35,50,70),(x1,y1),(x2,y2),max(2,w//3))
    pygame.draw.line(s,(200,230,255),(x1,y1),(x2,y2),max(1,w//6))
    if water:
        cnt, off = max(3, int(cap*.6)), (tick*.012)%1
        for i in range(cnt):
            t = (off+i/cnt)%1; px, py = int(x1+dx*t), int(y1+dy*t)
            pygame.draw.circle(s,C["water"],(px,py),4); pygame.draw.circle(s,(200,240,255),(px,py),2)


def draw_label(s, x1, y1, x2, y2, cap, cut, minc):
    dx,dy = x2-x1,y2-y1; l = math.hypot(dx,dy) or 1; nx,ny = -dy/l, dx/l
    mx,my = (x1+x2)/2, (y1+y2)/2; lx,ly = int(mx+nx*36), int(my+ny*36)
    pygame.draw.line(s,(90,120,160),(int(mx),int(my)),(lx,ly),1); pygame.draw.circle(s,(90,120,160),(int(mx),int(my)),3)
    col = C["red"] if cut else C["yellow"] if minc else C["blue"]
    msg = f"{cap} l/s"; rect = pygame.Rect(lx-f(13,True).size(msg)[0]//2-7, ly-12, f(13,True).size(msg)[0]+14, 24)
    pygame.draw.rect(s,(2,5,14),rect,border_radius=5); pygame.draw.rect(s,col,rect,2,border_radius=5)
    text(s,msg,13,col,rect.center,True,True)


class App:
    def __init__(self):
        self.size = (W0,H0); self.screen = pygame.display.set_mode(self.size, pygame.RESIZABLE)
        pygame.display.set_caption("Max-Flow / Min-Cut - Hidranti dhe Depozita")
        self.clock = pygame.time.Clock(); self.tick = 0; self.full = False; self.gw = int(W0*.72)
        self.cut, self.mouse, self.hover = set(), (0,0), None
        self.show_help = self.show_min = False
        self.mf, _ = max_flow(); self.min_edges, _, _ = min_cut()
        self.btns = {}

    def positions(self): return node_positions(self.gw, self.screen.get_height())

    def edge_at(self, pos):
        p, best, dmin = self.positions(), None, 14
        for u,v,_ in EDGES:
            d = dist_point_line(pos[0],pos[1],*p[u],*p[v])
            if d < dmin: best, dmin = (u,v), d
        return best

    def sides(self):
        s_side, t_side = partition(self.cut)
        return s_side, t_side, SINK in t_side

    def water_level(self, n, s_side, valid):
        if n == SOURCE: return 1.0
        if n == SINK: return 1.0 if not self.cut else (0.05 if valid else 0.5)
        return 1.0 if (not self.cut or n in s_side) else 0.2

    def draw_graph(self):
        s, h, p = self.screen, self.screen.get_height(), self.positions()
        s_side, t_side, valid = self.sides()
        pygame.draw.rect(s,C["graph"],(12,80,self.gw-24,h-128),border_radius=16)
        pygame.draw.rect(s,(6,10,22),(12,4,self.gw-24,76),border_radius=10)
        text(s,"MAX-FLOW / MIN-CUT - Hidranti dhe Depozita",22,C["white"],(self.gw//2,24),True,True)
        text(s,"Kliko tubat për t'i prerë dhe gjej prerjen minimale",13,C["yellow"],(self.gw//2,50),True)
        for u,v,cap in EDGES:
            draw_pipe(s,*p[u],*p[v],cap,(u,v) in self.cut,self.hover==(u,v),u in s_side and (u,v) not in self.cut,self.tick)
        for u,v,cap in EDGES:
            draw_label(s,*p[u],*p[v],cap,(u,v) in self.cut,self.show_min and (u,v) in set(self.min_edges))
        for n,(x,y) in p.items():
            col = C["neutral"] if not valid else C["s"] if n in s_side else C["t"]
            draw_node(s,n,x,y,col,self.water_level(n,s_side,valid),self.tick)
        self.status_bar(s_side,t_side,valid)

    def status_bar(self, s_side, t_side, valid):
        if not self.cut: msg,bg,br = "Fillo duke klikuar tubat për t'i prerë.",(8,18,50),C["blue"]
        elif not valid: msg,bg,br = f"Prerja nuk është e vlefshme: hidranti ende arrin depozitën. ({len(self.cut)} tuba)",(48,24,8),C["orange"]
        else:
            cap = sum(c for u,v,c in EDGES if u in s_side and v in t_side)
            msg,bg,br = (f" Min-Cut = {cap} = Max-Flow.",(8,52,18),C["green"]) if cap == self.mf else (f"Prerje e vlefshme: kapaciteti = {cap}. Provo më të vogël.",(8,36,50),C["cyan"])
        r = pygame.Rect(12,self.screen.get_height()-46,self.gw-24,36)
        pygame.draw.rect(self.screen,bg,r,border_radius=8); pygame.draw.rect(self.screen,br,r,1,border_radius=8)
        text(self.screen,msg,13,C["white"],r.center,True)

    def button(self, key, label, col, rect, active=False):
        self.btns[key] = rect; hover = rect.collidepoint(self.mouse)
        pygame.draw.rect(self.screen,col if active else C["btn_h"] if hover else C["btn"],rect,border_radius=6)
        pygame.draw.rect(self.screen,C["white"] if active else col,rect,2,border_radius=6)
        text(self.screen,label,11,(10,10,20) if active else col,rect.center,True,True)

    def draw_panel(self):
        s, H, px, pw, pad = self.screen, self.screen.get_height(), self.gw, self.screen.get_width()-self.gw, 12
        pygame.draw.rect(s,(8,13,28),(px,0,pw,H)); pygame.draw.rect(s,C["panel"],(px+6,6,pw-12,H-12),border_radius=12)
        bw, bh, gap, y = (pw-pad*2-12)//3, 32, 6, 14
        self.button("reset","RESET",C["orange"],pygame.Rect(px+pad,y,bw,bh))
        self.button("min","MIN-CUT ON" if self.show_min else "TREGO MIN-CUT",C["yellow"],pygame.Rect(px+pad+bw+gap,y,bw,bh),self.show_min)
        self.button("help","NDIHMË",C["cyan"],pygame.Rect(px+pad+(bw+gap)*2,y,bw,bh))
        y += bh+16
        text(s,"MAX-FLOW E LLOGARITUR",12,C["dim"],(px+pw//2,y),True,True); y += 20
        text(s,self.mf,44,C["green"],(px+pw//2,y+20),True,True); y += 50
        text(s,"l / sek",11,C["dim"],(px+pw//2,y),True); y += 28
        pygame.draw.line(s,(30,45,75),(px+pad,y),(px+pw-pad,y)); y += 12
        y = self.cut_panel(px,pw,y,pad)
        self.edge_list(px,pw,y,pad)
        self.controls(px,pw,H,pad)
        if self.show_help: self.help_overlay()

    def cut_panel(self, px, pw, y, pad):
        s_side,t_side,valid = self.sides(); clicked = sum(c for u,v,c in EDGES if (u,v) in self.cut)
        text(self.screen,"PRERJA JOTE (CUT)",12,C["dim"],(px+pad,y),bold=True); y += 22
        for label,val in [("Tuba të prerë:",len(self.cut)),("Kap. e klikuar:",clicked)]:
            text(self.screen,label,11,C["dim"],(px+pad,y)); text(self.screen,val,13,C["white"],(px+pad+120,y-1),bold=True); y += 18
        y += 4
        cap = sum(c for u,v,c in EDGES if u in s_side and v in t_side)
        stat,col = ("Asnjë tub i prerë",C["dim"]) if not self.cut else ("✗ JO E VLEFSHME",C["orange"]) if not valid else (" MIN-CUT!",C["green"]) if cap == self.mf else ("✓ E VLEFSHME",C["cyan"])
        r = pygame.Rect(px+pad,y,pw-pad*2,24); pygame.draw.rect(self.screen,(8,12,24),r,border_radius=4); pygame.draw.rect(self.screen,col,r,1,border_radius=4)
        text(self.screen,stat,12,col,r.center,True,True); y += 32
        if valid and self.cut:
            text(self.screen,"Partition:",11,C["dim"],(px+pad,y)); y += 18
            text(self.screen,"S = {"+", ".join(NAMES[n] for n in sorted(s_side))+"}",10,C["s"],(px+pad+6,y)); y += 14
            text(self.screen,"T = {"+", ".join(NAMES[n] for n in sorted(t_side))+"}",10,C["t"],(px+pad+6,y)); y += 20
            text(self.screen,f"Harqe S→T: {sum(1 for u,v,_ in EDGES if u in s_side and v in t_side)}",11,C["dim"],(px+pad,y)); y += 16
            text(self.screen,"Kapaciteti real:",11,C["dim"],(px+pad,y)); text(self.screen,cap,13,C["green"],(px+pad+120,y-1),bold=True); y += 22
            if cap == self.mf:
                r = pygame.Rect(px+pad,y,pw-pad*2,36); pygame.draw.rect(self.screen,(8,48,18),r,border_radius=6); pygame.draw.rect(self.screen,C["green"],r,2,border_radius=6)
                text(self.screen,"MIN-CUT = MAX-FLOW",11,C["green"],(px+pw//2,y+12),True,True); text(self.screen,"Teorema themelore",9,(180,255,180),(px+pw//2,y+26),True); y += 44
        pygame.draw.line(self.screen,(30,45,75),(px+pad,y),(px+pw-pad,y)); return y+10

    def edge_list(self, px, pw, y, pad):
        text(self.screen,"LISTA E TUBAVE",12,C["dim"],(px+pad,y),bold=True); y += 20
        for u,v,cap in EDGES:
            if y > self.screen.get_height()-80: break
            cut, mini = (u,v) in self.cut, self.show_min and (u,v) in set(self.min_edges)
            col, mark = (C["red"],"✗") if cut else (C["yellow"],"◆") if mini else (C["dim"],"─")
            text(self.screen,f"{mark} {NAMES[u]}→{NAMES[v]:<8} {cap} l/s",10,col,(px+pad+4,y)); y += 14

    def controls(self, px, pw, H, pad):
        y = H-56; pygame.draw.line(self.screen,(30,45,75),(px+pad,y),(px+pw-pad,y)); y += 6
        for line in ["Klik tub: prer/restauro", "M: trego min-cut  |  H: ndihmë", "R: reset  |  ESC: dil"]:
            text(self.screen,line,10,(65,85,130),(px+pad+2,y)); y += 13

    def help_overlay(self):
        s, W, H = self.screen, *self.screen.get_size()
        ov = pygame.Surface((W,H),pygame.SRCALPHA); ov.fill((0,0,0,180)); s.blit(ov,(0,0))
        bw,bh = 700,430; x,y = (W-bw)//2,(H-bh)//2
        pygame.draw.rect(s,(16,24,48),(x,y,bw,bh),border_radius=12); pygame.draw.rect(s,C["cyan"],(x,y,bw,bh),2,border_radius=12)
        text(s,"NDIHMË - Max-Flow / Min-Cut",22,C["cyan"],(W//2,y+30),True,True)
        lines = ["Problemi: sa ujë mund të rrjedhë maksimalisht nga Hidranti te Depozita?", "", "Cut është një grup tubash që e ndan burimin nga kulmi.", "Kapaciteti i cut-it është shuma e tubave që kalojnë nga S te T.", "", "Teorema: Max-Flow = Min-Cut.", "", "1. Kliko tubat për t'i prerë.", "2. Shiko nëse Hidranti nuk arrin më te Depozita.", "3. Provo të gjesh prerjen me kapacitet më të vogël.", "4. Kur kapaciteti bëhet sa Max-Flow, ke gjetur Min-Cut.", "", "Kliko kudo për ta mbyllur këtë mesazh."]
        for i,line in enumerate(lines): text(s,line,13,C["white"],(x+30,y+75+i*21))

    def events(self):
        for e in pygame.event.get():
            if e.type == pygame.QUIT: return False
            if e.type == pygame.VIDEORESIZE and not self.full:
                self.size = (e.w,e.h); self.screen = pygame.display.set_mode(self.size, pygame.RESIZABLE); self.gw = int(e.w*.72)
            if e.type == pygame.MOUSEMOTION:
                self.mouse = e.pos; self.hover = self.edge_at(e.pos) if e.pos[0] < self.gw else None
            if e.type == pygame.MOUSEBUTTONDOWN and e.button == 1:
                if self.show_help: self.show_help = False; continue
                if self.btns.get("reset", pygame.Rect(0,0,0,0)).collidepoint(e.pos): self.cut.clear(); self.show_min = False
                elif self.btns.get("min", pygame.Rect(0,0,0,0)).collidepoint(e.pos): self.show_min = not self.show_min
                elif self.btns.get("help", pygame.Rect(0,0,0,0)).collidepoint(e.pos): self.show_help = True
                else:
                    edge = self.edge_at(e.pos)
                    if edge: self.cut.remove(edge) if edge in self.cut else self.cut.add(edge); self.show_min = False
            if e.type == pygame.KEYDOWN:
                if e.key == pygame.K_ESCAPE: return False if not self.show_help else self.close_help()
                if e.key == pygame.K_r: self.cut.clear(); self.show_min = False
                if e.key == pygame.K_m: self.show_min = not self.show_min
                if e.key == pygame.K_h: self.show_help = not self.show_help
                if e.key in (pygame.K_f, pygame.K_F11): self.toggle_full()
        return True

    def close_help(self): self.show_help = False; return True

    def toggle_full(self):
        self.full = not self.full
        self.screen = pygame.display.set_mode((0,0), pygame.FULLSCREEN) if self.full else pygame.display.set_mode(self.size, pygame.RESIZABLE)
        self.gw = int(self.screen.get_width()*.72)

    def draw(self):
        self.screen.fill(C["bg"]); self.draw_graph(); self.draw_panel(); pygame.display.flip()

    def run(self):
        while self.events():
            self.tick += 1; self.draw(); self.clock.tick(60)
        pygame.quit()


def print_intro():
    mf, _ = max_flow(); edges, _, s_side = min_cut(); cap = sum(c for u,v,c in EDGES if (u,v) in set(edges))
    print("\n" + "="*70)
    print("  USHTRIMI 0 - MAX-FLOW / MIN-CUT")
    print("  Hidranti -> Brryla tubash -> Depozita")
    print("="*70)
    print(f"\n  Grafi: {len(NAMES)} nyje, {len(EDGES)} tuba")
    print(f"  Max-Flow = {mf} l/s")
    print("\n  Min-Cut teorik:")
    print(f"    Tubat: {[(NAMES[u], NAMES[v]) for u,v in edges]}")
    print(f"    Kapaciteti: {cap} l/s")
    print(f"    S-side: {sorted(NAMES[n] for n in s_side)}")
    print("="*70 + "\n")


if __name__ == "__main__":
    print_intro()
    App().run()



  USHTRIMI 0 - MAX-FLOW / MIN-CUT
  Hidranti -> Brryla tubash -> Depozita

  Grafi: 6 nyje, 7 tuba
  Max-Flow = 14 l/s

  Min-Cut teorik:
    Tubat: [('B-1', 'B-3'), ('B-4', 'Depozita')]
    Kapaciteti: 14 l/s
    S-side: ['B-1', 'B-2', 'B-4', 'Hidranti']



---

## Ushtrimi 2 — Ford-Fulkerson vs Edmonds-Karp: Vizualizim Hap-pas-Hapi

#### Ford-Fulkerson (DFS)
Algoritmi bazë i rrjedhës maksimal:
1. Gjej një rrugë rritëse nga S te T duke përdorur **DFS**
2. Gjej **ngushticën** (kapacitetin minimal) në atë rrugë
3. Përditëso rrjedhën dhe **graf-in mbetës**
4. Përsërit deri sa nuk ka më rrugë rritëse

**Graf mbetës:** Pas çdo përditësimi, çdo brinjë `u→v` me rrjedhën `f`:
- Kapaciteti mbetës `u→v` = `c(u,v) - f(u,v)` (sa mbetet)
- Shtohet brinja e kthimit `v→u` me kapacitet `f(u,v)` (mund të "anulosh" rrjedhën)

#### Edmonds-Karp (BFS)
- **E njëjtë** me Ford-Fulkerson, por përdor **BFS** për të gjetur rrugën **më të shkurtër**
- Garanton kompleksitet **O(V·E²)** — gjithmonë polinomial
- BFS gjen rrugën me numrin **minimal** të brinjëve

#### Krahasimi vizual:
- **DFS** mund të gjejë rrugë të gjata → më shumë iterime
- **BFS** gjithmonë gjejë rrugën më të shkurtër → pak iterime

---

### Skenari Real: Rrjeti i Ujësjellësit të Qytetit

Imagjino se je inxhinier i ujësjellësit të një qyteti. Keni:
- **Burim (S):** Stacioni kryesor i pompimit të ujit
- **Kulm (T):** Rezervuari qendror i qytetit
- **Nyjet 0, 1, 2, 3:** Stacione pompimi ndërmjetëse
- **Kapacitetet:** Sasi e ujit (litra/sekondë) që mund të kalojë çdo linjë

**Pyetja:** Sa ujë maksimal mund të pompojmë nga stacioni te rezervuari?

#### Modelimi si Graf:
```
       [10]         [25]
  S ─────────> [0] ─────────> T
  │             │ ↗            ↑
  │[10]     [6] │          [10]│
  │             ↓              │
  └──────────> [1] ─────────> [3]
        [15]         [6]↗
                         [3]  
```

#### Zgjidhja me algoritëm:
Ford-Fulkerson dhe Edmonds-Karp do të gjejnë rrugë rritëse dhe do të llogarisim **rrjedhën totale** që arrin te rezervuari. Vizualizimi tregon **çdo hap** të algoritmit.


In [ ]:
from collections import deque

import plotly.graph_objects as go
from IPython.display import Javascript, display
from ipywidgets import Button, HBox, HTML, Output, VBox


NYJET = {
    0: ("S", "Burimi"),
    1: ("0", "Nyja 0"),
    2: ("1", "Nyja 1"),
    3: ("2", "Nyja 2"),
    4: ("3", "Nyja 3"),
    5: ("T", "Kulmi"),
}

BRINJET = [
    (0, 1, 10),  # S -> 0
    (0, 2, 10),  # S -> 1
    (1, 3, 25),  # 0 -> 2
    (2, 4, 15),  # 1 -> 3
    (3, 5, 10),  # 2 -> T
    (4, 1, 6),   # 3 -> 0
    (4, 5, 10),  # 3 -> T
]

POZICIONET = {
    0: (0, 0),
    1: (2, 1.2),
    2: (2, -1.2),
    3: (4, 1.2),
    4: (4, -1.2),
    5: (6, 0),
}

BURIMI, KULMI = 0, 5

NGJYRA = {
    "sfond": "#0f172a",
    "burim": "#2563eb",
    "kulm": "#dc2626",
    "nyje": "#16a34a",
    "brinje": "#FFFFFF",
    "rrjedhe": "#22c55e",
    "eksploruar": "#facc15",
    "rruge": "#00d9ff",
    "panel": "#1f2937",
    "kornize": "#38bdf8",
}

RENDI_DFS = {
    0: [2, 1],
    1: [3],
    2: [4],
    3: [5],
    4: [1, 5],
    5: [],
}


class RrjedhaMaksimale:
    """Llogarit hapat e Ford-Fulkerson (DFS) ose Edmonds-Karp (BFS)."""

    def __init__(self, nr_nyje, brinjet, burimi, kulmi, strategjia):
        self.nr_nyje = nr_nyje
        self.brinjet = brinjet
        self.burimi = burimi
        self.kulmi = kulmi
        self.strategjia = strategjia
        self.kapaciteti = [[0] * nr_nyje for _ in range(nr_nyje)]
        self.rrjedha = [[0] * nr_nyje for _ in range(nr_nyje)]
        self.hapat = []

        for u, v, c in brinjet:
            self.kapaciteti[u][v] = c

        self._ekzekuto()

    @property
    def emri_algoritmit(self):
        return "Ford-Fulkerson" if self.strategjia == "dfs" else "Edmonds-Karp"

    @property
    def emri_kerkimit(self):
        return "DFS" if self.strategjia == "dfs" else "BFS"

    def _kopjo_rrjedhen(self):
        return [rresht[:] for rresht in self.rrjedha]

    def _rezidual(self, u, v):
        return self.kapaciteti[u][v] - self.rrjedha[u][v]

    def _kerkim_dfs(self):
        vizituar = [False] * self.nr_nyje
        prindi = [-1] * self.nr_nyje
        eksploruar = []

        def dfs(u):
            vizituar[u] = True
            if u == self.kulmi:
                return True

            for v in RENDI_DFS[u]:
                if not vizituar[v] and self._rezidual(u, v) > 0:
                    prindi[v] = u
                    eksploruar.append((u, v))
                    if dfs(v):
                        return True
            return False

        return dfs(self.burimi), prindi, eksploruar

    def _kerkim_bfs(self):
        vizituar = [False] * self.nr_nyje
        prindi = [-1] * self.nr_nyje
        eksploruar = []
        radha = deque([self.burimi])
        vizituar[self.burimi] = True

        while radha:
            u = radha.popleft()
            for v in range(self.nr_nyje):
                if not vizituar[v] and self._rezidual(u, v) > 0:
                    vizituar[v] = True
                    prindi[v] = u
                    eksploruar.append((u, v))
                    if v == self.kulmi:
                        return True, prindi, eksploruar
                    radha.append(v)

        return False, prindi, eksploruar

    def _kerko_rruge(self):
        return self._kerkim_bfs() if self.strategjia == "bfs" else self._kerkim_dfs()

    def _nderto_rrugen(self, prindi):
        rruga = []
        v = self.kulmi
        while v != self.burimi:
            u = prindi[v]
            rruga.append((u, v))
            v = u
        return list(reversed(rruga))

    def _shto_hap(self, lloji, titulli, pershkrimi, rrjedha_totale,
                  iteracioni, rruga=None, eksploruar=None, ngushtica=None):
        self.hapat.append({
            "lloji": lloji,
            "titulli": titulli,
            "pershkrimi": pershkrimi,
            "rrjedha": self._kopjo_rrjedhen(),
            "rruga": rruga or [],
            "eksploruar": eksploruar or [],
            "ngushtica": ngushtica,
            "rrjedha_totale": rrjedha_totale,
            "iteracioni": iteracioni,
            "algoritmi": self.emri_algoritmit,
            "kerkimi": self.emri_kerkimit,
        })

    def _ekzekuto(self):
        rrjedha_totale = 0
        iteracioni = 0

        self._shto_hap(
            "init",
            f"{self.emri_algoritmit}: grafi fillestar",
            "Të gjitha rrjedhat fillojnë nga 0.",
            rrjedha_totale,
            iteracioni,
        )

        while True:
            iteracioni += 1
            u_gjet, prindi, eksploruar = self._kerko_rruge()

            self._shto_hap(
                self.strategjia,
                f"Iteracioni {iteracioni}: kërkim me {self.emri_kerkimit}",
                f"{self.emri_kerkimit} kërkon një rrugë rritëse në graf-in rezidual.",
                rrjedha_totale,
                iteracioni,
                eksploruar=eksploruar,
            )

            if not u_gjet:
                self._shto_hap(
                    "done",
                    f"{self.emri_algoritmit}: algoritmi përfundoi",
                    "Nuk ekziston më rrugë rritëse.",
                    rrjedha_totale,
                    iteracioni,
                )
                break

            rruga = self._nderto_rrugen(prindi)
            ngushtica = min(self._rezidual(u, v) for u, v in rruga)

            self._shto_hap(
                "augment",
                f"Iteracioni {iteracioni}: u gjet rruga rritëse",
                "Rruga e theksuar mund të rrisë rrjedhën totale.",
                rrjedha_totale,
                iteracioni,
                rruga=rruga,
                ngushtica=ngushtica,
            )

            for u, v in rruga:
                self.rrjedha[u][v] += ngushtica
                self.rrjedha[v][u] -= ngushtica

            rrjedha_totale += ngushtica

            self._shto_hap(
                "update",
                f"Iteracioni {iteracioni}: rrjedha u përditësua",
                f"Vlera e ngushticës {ngushtica} u shtua në rrugën e zgjedhur.",
                rrjedha_totale,
                iteracioni,
                rruga=rruga,
                ngushtica=ngushtica,
            )


ford_fulkerson = RrjedhaMaksimale(len(NYJET), BRINJET, BURIMI, KULMI, "dfs")
edmonds_karp = RrjedhaMaksimale(len(NYJET), BRINJET, BURIMI, KULMI, "bfs")


# Vizualizimi i grafit dhe hapave

def stili_i_brinjes(hapi, u, v, rrjedha_aktuale):
    if (u, v) in set(hapi["rruga"]):
        return NGJYRA["rruge"], 8
    if (u, v) in set(hapi["eksploruar"]):
        return NGJYRA["eksploruar"], 6
    if rrjedha_aktuale > 0:
        return NGJYRA["rrjedhe"], 6
    return NGJYRA["brinje"], 4


def vizato_brinjet(hapi):
    traces = []
    for u, v, kapaciteti in BRINJET:
        x1, y1 = POZICIONET[u]
        x2, y2 = POZICIONET[v]
        rrjedha_aktuale = max(0, hapi["rrjedha"][u][v])
        ngjyra, trashesia = stili_i_brinjes(hapi, u, v, rrjedha_aktuale)
        traces.append(go.Scatter(
            x=[x1, x2], y=[y1, y2], mode="lines",
            line=dict(width=trashesia, color=ngjyra),
            hoverinfo="none", showlegend=False,
        ))
    return traces


def vizato_nyjet():
    x, y, etiketa, hover, ngjyra = [], [], [], [], []

    for nyja, (label, emer) in NYJET.items():
        px, py = POZICIONET[nyja]
        x.append(px)
        y.append(py)
        etiketa.append(label)
        hover.append(emer)
        ngjyra.append(
            NGJYRA["burim"] if nyja == BURIMI
            else NGJYRA["kulm"] if nyja == KULMI
            else NGJYRA["nyje"]
        )

    return go.Scatter(
        x=x, y=y, mode="markers+text", text=etiketa,
        hovertext=hover, hoverinfo="text", textposition="middle center",
        textfont=dict(size=22, color="white"),
        marker=dict(size=58, color=ngjyra, line=dict(width=3, color="white")),
        showlegend=False,
    )


def shenimet(hapi):
    anotime = []

    for u, v, kapaciteti in BRINJET:
        x1, y1 = POZICIONET[u]
        x2, y2 = POZICIONET[v]
        rrjedha_aktuale = max(0, hapi["rrjedha"][u][v])
        ngjyra, _ = stili_i_brinjes(hapi, u, v, rrjedha_aktuale)
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2

        anotime += [
            dict(
                ax=x1, ay=y1, x=x2, y=y2,
                xref="x", yref="y", axref="x", ayref="y",
                showarrow=True, arrowhead=3, arrowsize=1.5,
                arrowwidth=2.5, arrowcolor=ngjyra, opacity=0.95,
            ),
            dict(
                x=mx, y=my + 0.18, xref="x", yref="y",
                text=f"<b>{rrjedha_aktuale}/{kapaciteti}</b>",
                showarrow=False, font=dict(size=15, color=ngjyra),
                bgcolor="#111827", bordercolor=ngjyra,
                borderwidth=1, borderpad=4,
            ),
        ]

    anotime.append(dict(
        x=7.2, y=1.6, xref="x", yref="y", showarrow=False,
        align="left", font=dict(size=15, color="white"),
        bgcolor=NGJYRA["panel"], bordercolor=NGJYRA["kornize"],
        borderwidth=2, borderpad=12,
        text=(
            f"<b>{hapi['algoritmi']} ({hapi['kerkimi']})</b><br>"
            f"<b>{hapi['titulli']}</b><br>"
            f"{hapi['pershkrimi']}<br><br>"
            f"<b>Rrjedha maksimale:</b> {hapi['rrjedha_totale']}<br>"
            f"<b>Faza:</b> {hapi['lloji'].upper()}<br>"
            f"<b>Ngushtica:</b> {hapi['ngushtica'] if hapi['ngushtica'] is not None else '-'}"
        ),
    ))

    return anotime


def krijo_figuren(hapi):
    figura = go.Figure(data=vizato_brinjet(hapi) + [vizato_nyjet()])
    figura.update_layout(
        title=dict(
            text=f"{hapi['algoritmi']} ({hapi['kerkimi']}) — {hapi['lloji'].upper()}",
            font=dict(size=24, color="white"),
        ),
        width=1500,
        height=500,
        paper_bgcolor=NGJYRA["sfond"],
        plot_bgcolor=NGJYRA["sfond"],
        xaxis=dict(visible=False, range=[-0.5, 8]),
        yaxis=dict(visible=False, range=[-2.2, 2.2]),
        margin=dict(l=30, r=30, t=80, b=30),
        annotations=shenimet(hapi),
    )
    return figura


algoritmi_aktiv = "ff"
indeksi_hapit = 0
outputi = Output()
statusi = HTML()

butoni_ff = Button(description="Ford-Fulkerson (DFS)", button_style="info")
butoni_ek = Button(description="Edmonds-Karp (BFS)")
butoni_pasambrapa = Button(description="◀ Previous")
butoni_para = Button(description="Next ▶")


def hapat_aktuale():
    return ford_fulkerson.hapat if algoritmi_aktiv == "ff" else edmonds_karp.hapat


def shfaq_hapin():
    hapi = hapat_aktuale()[indeksi_hapit]
    statusi.value = (
        f"<h3 style='color:#facc15; margin:4px 0;'>"
        f"{hapi['algoritmi']} ({hapi['kerkimi']}) — Hapi {indeksi_hapit + 1}/{len(hapat_aktuale())}</h3>"
        f"<div style='color:white;'>"
        f"<b>Faza:</b> {hapi['lloji'].upper()} | "
        f"<b>Rrjedha maksimale:</b> {hapi['rrjedha_totale']} | "
        f"<b>Ngushtica:</b> {hapi['ngushtica'] if hapi['ngushtica'] is not None else '-'}"
        f"</div>"
    )

    butoni_ff.button_style = "info" if algoritmi_aktiv == "ff" else ""
    butoni_ek.button_style = "info" if algoritmi_aktiv == "ek" else ""
    butoni_pasambrapa.disabled = indeksi_hapit == 0
    butoni_para.disabled = indeksi_hapit == len(hapat_aktuale()) - 1

    with outputi:
        outputi.clear_output(wait=True)
        display(krijo_figuren(hapi))


def zgjidh_algoritmin(emri):
    global algoritmi_aktiv, indeksi_hapit
    algoritmi_aktiv = emri
    indeksi_hapit = 0
    shfaq_hapin()


def ndrysho_hapin(ndryshimi):
    global indeksi_hapit
    kufiri = len(hapat_aktuale()) - 1
    indeksi_hapit = max(0, min(kufiri, indeksi_hapit + ndryshimi))
    shfaq_hapin()


butoni_ff.on_click(lambda _: zgjidh_algoritmin("ff"))
butoni_ek.on_click(lambda _: zgjidh_algoritmin("ek"))
butoni_pasambrapa.on_click(lambda _: ndrysho_hapin(-1))
butoni_para.on_click(lambda _: ndrysho_hapin(1))

display(VBox([HBox([butoni_ff, butoni_ek, butoni_pasambrapa, butoni_para]), statusi, outputi]))
shfaq_hapin()

# Shkurtore tastiere për notebook: ←, →, 1, 2
display(Javascript("""
(function() {
    if (window.__maxflowKeyboardHandlerInstalled) return;
    window.__maxflowKeyboardHandlerInstalled = true;

    document.addEventListener('keydown', function(e) {
        if (!['ArrowRight', 'ArrowLeft', '1', '2'].includes(e.key)) return;
        const buttons = Array.from(document.querySelectorAll('button'));
        const click = text => {
            const btn = buttons.find(b => b.innerText.trim() === text);
            if (btn) btn.click();
        };
        e.preventDefault();
        if (e.key === 'ArrowRight') click('Next ▶');
        if (e.key === 'ArrowLeft') click('◀ Previous');
        if (e.key === '1') click('Ford-Fulkerson (DFS)');
        if (e.key === '2') click('Edmonds-Karp (BFS)');
    });
})();
"""))


<IPython.core.display.Javascript object>

---

## Ushtrimi 3 — Network Capacity Planning: Ford-Fulkerson vs Edmonds-Karp

#### Pse Max-Flow në rrjetet kompjuterike?
Rrjedha maksimal është thelbesore në:
- **Planifikimi i kapacitetit rrjetor** (network capacity planning)
- **Load balancing** — shpërndarja e barabartë e trafikut
- **Fault tolerance** — gjejë ku rrjeti është i cenueshëm
- **CDN routing** — si shpërndahen videot te përdoruesit

#### Bottleneck Analysis:
> Brinja e ngushticës (bottleneck) është ajo që **kufizon** rrjedhën total. Zëvendësimi i saj ka **impaktin më të madh** në performancë.

#### Graf-i Rezidual — Vizualizim:
```
Fillimisht:           Pas f(DC→R-A)=5:
DC ──[12]──> R-A      DC ──[7]──> R-A
                       DC <──[5]── R-A  (brinja e kthimit!)
```

#### Pse BFS (Edmonds-Karp) është më i mirë?
- Gjen **rrugën më të shkurtër** → kapacitet rezidual më i madh zakonisht
- Numri i iterimeve ≤ O(V·E) — garanci matematikore
- DFS mund të zgjedhë rrugë të gjata me bottleneck të vogël

---

### Skenari Real: Data Center Network

Imagjino se je arkitekt rrjeti për një kompani të madhe. Ke:
- **Data Center (DC):** Burimi i të dhënave (serverët kryesorë)
- **Central Server (SRV):** Destinacioni (platforma e klientëve)
- **Core Routers (R-A, R-B):** Routerët kryesorë — lidhin DC me switch-et
- **Distribution Switches (SW-1, SW-2, SW-3):** Shpërndarës ndërmjetës

**Pyetja:** Sa Gbps maksimale mund të dërgojmë nga Data Center te serveri qendror?

#### Modelimi si Graf:
```
         R-A ──[8]──> SW-1 ──[9]──> SRV
        ↗ ↑[4]↘  [3]↗     ↘[12]↗  ↑
[12]   /   |   SW-2          SW-3──[8]
DC          |[7]↗  ↘[5]       ↑[10]
      [15]↘ |       ↘        /
        R-B ─[6]──> SW-3
```

#### Cfarë do të analizojmë:
1. **Rrjedha maksimale** — throughput total i rrjetit
2. **Bottleneck edges** — ku duhet të investojmë për kapacitet më të lartë
3. **Krahasimi DFS vs BFS** — cilat rrugë zgjedh secili algoritëm
4. **Numri i iterimeve** — eficienca e secilit algoritëm

> **Rezultati praktik:** Pas ekzekutimit, do të shohim se cila lidhje është "ngushtim" i rrjetit — aty duhet të investojmë për të rritur performancën!


In [ ]:
import pygame, math, random
from collections import defaultdict, deque

pygame.init()

W0, H0 = 1440, 860
SOURCE, SINK = 0, 6
EDGES = [
    (0,1,12),(0,2,15),(1,2,4),(1,3,8),(1,4,7),(2,4,6),
    (2,5,10),(3,4,3),(4,5,5),(3,6,9),(4,6,12),(5,6,8)
]
NAMES = {0:"DC",1:"R-A",2:"R-B",3:"SW-1",4:"SW-2",5:"SW-3",6:"SRV"}
FULL = {0:"Data Center",1:"Core Router A",2:"Core Router B",3:"Distribution Switch 1",4:"Distribution Switch 2",5:"Distribution Switch 3",6:"Central Server"}

C = {
    "bg":(8,14,28), "graph":(10,18,35), "panel":(15,24,46), "white":(230,245,255),
    "dim":(125,148,190), "blue":(70,190,255), "cyan":(40,220,255), "yellow":(240,190,40),
    "red":(255,80,55), "green":(60,220,120), "orange":(255,160,60), "link":(75,95,130),
    "link_dark":(20,30,48), "packet":(110,245,255), "spark":(255,250,200),
    "btn":(22,34,60), "btn_h":(32,48,80)
}
FONTS = {}


def font(size, bold=False):
    key = (size, bold)
    if key not in FONTS:
        FONTS[key] = pygame.font.SysFont("Segoe UI", size, bold=bold)
    return FONTS[key]


def text(s, msg, size, col, pos, center=False, bold=False):
    img = font(size, bold).render(str(msg), True, col)
    r = img.get_rect(center=pos) if center else img.get_rect(topleft=pos)
    s.blit(img, r)
    return r


def glow(s, x, y, col, radius):
    for r in range(radius, 0, -8):
        surf = pygame.Surface((2*r, 2*r), pygame.SRCALPHA)
        pygame.draw.circle(surf, (*col, int(70*r/radius)), (r, r), r)
        s.blit(surf, (x-r, y-r))


class MaxFlow:
    def __init__(self, strategy="dfs"):
        self.strategy = strategy
        self.cap = defaultdict(lambda: defaultdict(int))
        self.adj = defaultdict(set)
        self.flow = defaultdict(lambda: defaultdict(int))
        self.steps = []
        for u, v, c in EDGES:
            self.cap[u][v] += c
            self.adj[u].add(v)
            self.adj[v].add(u)
        self.run()

    def snap(self):
        return {u: dict(self.flow[u]) for u in self.flow}

    def search(self):
        return self.bfs() if self.strategy == "bfs" else self.dfs()

    def dfs(self):
        explored, seen = [], {SOURCE}

        def go(u, path):
            if u == SINK:
                return path
            for v in sorted(self.adj[u]):
                if v not in seen and self.cap[u][v] - self.flow[u][v] > 0:
                    seen.add(v)
                    explored.append((u, v))
                    found = go(v, path + [(u, v)])
                    if found:
                        return found
            return None

        return go(SOURCE, []), explored

    def bfs(self):
        parent, seen, q, explored = {SOURCE: None}, {SOURCE}, deque([SOURCE]), []
        while q:
            u = q.popleft()
            for v in sorted(self.adj[u]):
                if v not in seen and self.cap[u][v] - self.flow[u][v] > 0:
                    seen.add(v)
                    parent[v] = u
                    explored.append((u, v))
                    if v == SINK:
                        path, cur = [], SINK
                        while cur != SOURCE:
                            prev = parent[cur]
                            path.append((prev, cur))
                            cur = prev
                        return list(reversed(path)), explored
                    q.append(v)
        return None, explored

    def total(self):
        return sum(self.flow[SOURCE][v] for v in self.adj[SOURCE])

    def add_step(self, kind, **data):
        data.update({"type": kind, "flow_snap": self.snap(), "max_flow": self.total()})
        self.steps.append(data)

    def run(self):
        iteration = 0
        while True:
            iteration += 1
            path, explored = self.search()
            self.add_step(self.strategy, iteration=iteration, explored=explored, found=path is not None)
            if not path:
                self.add_step("done")
                break

            bottleneck = min(self.cap[u][v] - self.flow[u][v] for u, v in path)
            bottle_edge = min(path, key=lambda e: self.cap[e[0]][e[1]] - self.flow[e[0]][e[1]])
            nodes = [SOURCE] + [v for _, v in path]
            self.add_step("augment", iteration=iteration, path=path, path_nodes=nodes, bottle_edge=bottle_edge, bottleneck=bottleneck)

            for u, v in path:
                self.flow[u][v] += bottleneck
                self.flow[v][u] -= bottleneck
            self.add_step("update", iteration=iteration, path=path, bottle_edge=bottle_edge, bottleneck=bottleneck)


def node_positions(graph_w, h):
    top, bot = 90, h - 65
    span = bot - top
    return {
        0:(int(graph_w*.08), int(top+span*.50)),
        1:(int(graph_w*.30), int(top+span*.22)),
        2:(int(graph_w*.30), int(top+span*.78)),
        3:(int(graph_w*.58), int(top+span*.15)),
        4:(int(graph_w*.58), int(top+span*.50)),
        5:(int(graph_w*.58), int(top+span*.85)),
        6:(int(graph_w*.88), int(top+span*.50)),
    }


def draw_data_center(s, x, y, active, tick):
    if active:
        glow(s, x, y, C["cyan"], 90)
    pygame.draw.rect(s, (35,48,75), (x-52,y-40,104,80), border_radius=6)
    pygame.draw.rect(s, (120,170,210), (x-52,y-40,104,80), 2, border_radius=6)
    for i, ox in enumerate([-32, 0, 32]):
        pygame.draw.rect(s, (18,26,45), (x+ox-13,y-30,26,60), border_radius=3)
        for j in range(5):
            yy = y - 22 + j * 11
            led = C["green"] if active and (tick+i*7+j*3) % 28 < 16 else (55,85,120)
            pygame.draw.circle(s, led, (x+ox-6, yy), 2)
            pygame.draw.rect(s, (70,105,140), (x+ox-1, yy-2, 9, 3))
    text(s, "DC", 16, C["white"], (x, y+52), True, True)
    text(s, "Data Center", 11, C["cyan"], (x, y+68), True)


def draw_router(s, x, y, name, active, tick):
    if active:
        glow(s, x, y, C["yellow"], 75)
    pygame.draw.rect(s, (42,58,85), (x-44,y-28,88,56), border_radius=8)
    pygame.draw.rect(s, (155,185,220), (x-44,y-28,88,56), 2, border_radius=8)
    for i, ox in enumerate([-26,-12,2,16,30]):
        led = C["green"] if active and (tick+i*5) % 24 < 14 else (55,85,110)
        pygame.draw.circle(s, led, (x+ox, y-8), 4)
    for sx in [-1, 1]:
        pygame.draw.line(s, C["dim"], (x+26*sx, y-28), (x+40*sx, y-48), 2)
    if active:
        pygame.draw.arc(s, C["packet"], (x-58,y-62,36,28), 0, math.pi, 2)
        pygame.draw.arc(s, C["packet"], (x+22,y-62,36,28), 0, math.pi, 2)
    text(s, name, 14, C["white"], (x, y), True, True)
    text(s, "Core Router", 10, C["dim"], (x, y+15), True)


def draw_switch(s, x, y, name, active, tick):
    if active:
        glow(s, x+10, y+10, C["green"], 85)
    pygame.draw.rect(s, (35,50,78), (x-42,y-28,105,75), border_radius=6)
    pygame.draw.rect(s, (120,150,180), (x-42,y-28,105,75), 2, border_radius=6)
    for row in range(3):
        for col in range(5):
            px, py = x - 30 + col * 20, y - 14 + row * 22
            led = C["green"] if active and (tick+row*8+col*4) % 32 < 18 else (55,75,100)
            pygame.draw.rect(s, (18,28,44), (px-7, py-5, 14, 10), border_radius=2)
            pygame.draw.circle(s, led, (px, py+9), 2)
    text(s, name, 14, C["white"], (x+10, y+5), True, True)
    text(s, "Dist. Switch", 14, C["white"], (x+10, y+20), True)


def draw_server(s, x, y, active, tick):
    if active:
        glow(s, x, y, C["red"], 90)
    pygame.draw.rect(s, (210,220,235), (x-45,y-42,90,84), border_radius=5)
    pygame.draw.rect(s, (145,165,190), (x-45,y-42,90,84), 2, border_radius=5)
    for i in range(4):
        yy = y - 28 + i * 18
        led = C["green"] if active and (tick+i*6) % 30 < 20 else (70,95,125)
        pygame.draw.rect(s, (34,48,74), (x-32, yy, 64, 10), border_radius=3)
        pygame.draw.circle(s, led, (x+24, yy+5), 3)
    pygame.draw.circle(s, C["blue"], (x, y+25), 10)
    text(s, "SRV", 9, C["white"], (x, y+25), True, True)
    text(s, "SRV - Server", 14, C["red"], (x, y+54), True, True)
    text(s, "DESTINACIONI", 10, C["orange"], (x, y+69), True, True)


def draw_node(s, node, x, y, active, tick):
    if node == SOURCE:
        draw_data_center(s, x, y, active, tick)
    elif node in (1, 2):
        draw_router(s, x, y, NAMES[node], active, tick)
    elif node in (3, 4, 5):
        draw_switch(s, x, y, NAMES[node], active, tick)
    else:
        draw_server(s, x, y, active, tick)


def edge_style(u, v, state):
    path = state.get("path", [])
    explored = state.get("explored_edges", [])
    bottle = state.get("bottle_edge")
    if (u, v) == bottle or (v, u) == bottle:
        return C["red"], 9
    if (u, v) in path or (v, u) in path:
        return C["cyan"], 8
    if (u, v) in explored or (v, u) in explored:
        return C["yellow"], 6
    return C["link"], 5


def draw_arrow(s, x1, y1, dx, dy, col, flow):
    angle = math.atan2(dy, dx)
    arrow_col = C["white"] if col == C["red"] else C["red"]
    for t in (0.30, 0.62):
        tx, ty = x1 + dx * (t + 0.05), y1 + dy * (t + 0.05)
        pts = [
            (int(tx), int(ty)),
            (int(tx - 10 * math.cos(angle - .50)), int(ty - 10 * math.sin(angle - .50))),
            (int(tx - 10 * math.cos(angle + .50)), int(ty - 10 * math.sin(angle + .50))),
        ]
        pygame.draw.polygon(s, arrow_col, pts)
        pygame.draw.polygon(s, (5,8,18), pts, 1)


def draw_link(s, x1, y1, x2, y2, flow, cap, col, base_w, tick, active):
    dx, dy = x2 - x1, y2 - y1
    length = math.hypot(dx, dy) or 1
    width = max(7, base_w + int(cap * .30))

    pygame.draw.line(s, C["link_dark"], (x1,y1), (x2,y2), width+10)
    pygame.draw.line(s, col, (x1,y1), (x2,y2), width)
    pygame.draw.line(s, (35,50,70), (x1,y1), (x2,y2), max(2, width//3))
    pygame.draw.line(s, (185,215,245), (x1,y1), (x2,y2), max(1, width//7))

    if active and flow > 0:
        count, off = max(4, int(flow*.5)), (tick*.012) % 1
        for i in range(count):
            t = (off + i / count) % 1
            sx, sy = int(x1 + dx*t), int(y1 + dy*t)
            pygame.draw.rect(s, C["packet"], (sx-4, sy-3, 8, 6), border_radius=2)
            pygame.draw.circle(s, C["spark"], (sx, sy), 2)
        for d in (-2, 2):
            nx, ny = -dy/length*d, dx/length*d
            pygame.draw.line(s, (170,240,255), (x1+nx,y1+ny), (x2+nx,y2+ny), 1)

    if col == C["red"] and tick % 8 < 4:
        for _ in range(3):
            t = random.random()
            sx, sy = int(x1+dx*t), int(y1+dy*t)
            pygame.draw.line(s, C["spark"], (sx,sy), (sx+random.randint(-3,3), sy+random.choice([-1,1])*random.randint(4,10)), 2)

    draw_arrow(s, x1, y1, dx, dy, col, flow)


def draw_label(s, x1, y1, x2, y2, flow, cap, col):
    dx, dy = x2 - x1, y2 - y1
    length = math.hypot(dx, dy) or 1
    mx, my = x1 + dx*.45, y1 + dy*.45
    lx, ly = int(mx - dy/length*38), int(my + dx/length*38)
    msg = f"{flow}/{cap} Gbps"
    w = font(12, True).size(msg)[0] + 14
    rect = pygame.Rect(lx-w//2, ly-12, w, 24)
    pygame.draw.line(s, (90,120,160), (int(mx),int(my)), (lx,ly), 1)
    pygame.draw.circle(s, (90,120,160), (int(mx),int(my)), 3)
    pygame.draw.rect(s, (2,5,14), rect, border_radius=5)
    pygame.draw.rect(s, col, rect, 2, border_radius=5)
    text(s, msg, 12, C["green"] if flow > 0 else (180,200,230), rect.center, True, True)


class App:
    def __init__(self):
        self.windowed = (W0, H0)
        self.screen = pygame.display.set_mode(self.windowed, pygame.RESIZABLE)
        self.clock = pygame.time.Clock()
        self.full = False
        self.strategy = "dfs"
        self.tick = self.counter = self.step_idx = self.max_flow = 0
        self.speed = 1
        self.paused = True
        self.graph_w = int(W0 * .72)
        self.mouse = (0, 0)
        self.btns = {}
        self.reset()

    @property
    def algo_label(self):
        return "DFS" if self.strategy == "dfs" else "BFS"

    @property
    def algo_full(self):
        return "Ford-Fulkerson" if self.strategy == "dfs" else "Edmonds-Karp"

    def caption(self):
        pygame.display.set_caption(f"{self.algo_full} ({self.algo_label}) - Network Capacity Planning")

    def reset(self):
        self.caption()
        self.mf = MaxFlow(self.strategy)
        self.step_idx, self.max_flow = 0, 0
        self.logs = []
        self.cur = {"type":"init", "flow_snap":{}, "path":[], "explored_edges":[], "bottle_edge":None, "bottleneck":0, "path_nodes":[], "max_flow":0, "iteration":0, "found":False}
        for msg, col in [
            (f"=== {self.algo_full} ({self.algo_label}) - Network Capacity Planning ===", C["blue"]),
            (f"Graf: {len(NAMES)} nyje, {len(EDGES)} lidhje rrjeti.", C["dim"]),
            ("Te gjitha rrjedhjet fillojne nga 0 Gbps.", C["dim"]),
            (("DFS gjen rruge - mund te jete e gjate!" if self.strategy == "dfs" else "BFS gjen rrugen me te shkurter."), C["yellow"] if self.strategy == "dfs" else C["cyan"]),
            ("SPACE: luaj/pauzo  |  S: hap  |  R: rinicio", C["dim"]),
        ]:
            self.log(msg, col)

    def log(self, msg, col):
        self.logs.append((msg, col))
        self.logs = self.logs[-22:]

    def positions(self):
        return node_positions(self.graph_w, self.screen.get_height())

    def flow(self, u, v):
        return max(0, self.cur.get("flow_snap", {}).get(u, {}).get(v, 0))

    def set_strategy(self, strategy):
        if strategy != self.strategy:
            self.strategy = strategy
            self.reset()

    def apply_step(self):
        if self.step_idx < 0:
            self.step_idx = 0
        if self.step_idx >= len(self.mf.steps):
            return
        step = self.mf.steps[self.step_idx]
        self.step_idx += 1
        self.cur = {
            "type": step["type"], "flow_snap": step["flow_snap"], "path": step.get("path", []),
            "explored_edges": step.get("explored", []), "bottle_edge": step.get("bottle_edge"),
            "bottleneck": step.get("bottleneck", 0), "path_nodes": step.get("path_nodes", []),
            "max_flow": step.get("max_flow", self.max_flow), "iteration": step.get("iteration", self.cur.get("iteration", 0)),
            "found": step.get("found", False),
        }
        self.max_flow = step.get("max_flow", self.max_flow)
        self.add_log_for(step)

    def add_log_for(self, step):
        kind, it = step["type"], step.get("iteration", 0)
        if kind in ("dfs", "bfs"):
            msg = f"[{it}] {self.algo_label}: " + (f"gjeti rruge! ({len(step['explored'])} lidhje)" if step["found"] else "nuk ka me rruge - PERFUNDOI.")
            self.log(msg, C["yellow"] if step["found"] else C["orange"])
        elif kind == "augment":
            path = " -> ".join(NAMES[n] for n in step["path_nodes"])
            u, v = step["bottle_edge"]
            self.log(f"[{it}] Rruga e {self.algo_label}: {path}", C["cyan"])
            self.log(f"     Bottleneck ({NAMES[u]}->{NAMES[v]}): {step['bottleneck']} Gbps", C["red"])
        elif kind == "update":
            self.log(f"[{it}] Perditeso: +{step['bottleneck']} Gbps. Totali = {step['max_flow']}", C["green"])
        elif kind == "done":
            self.log(f"=== {self.algo_full} PERFUNDOI ===", C["green"])
            self.log(f"Throughput Maksimal = {step['max_flow']} Gbps.", C["green"])
            self.paused = True

    def status(self):
        kind, it = self.cur["type"], self.cur.get("iteration", 0)
        if kind == "init": return f"{self.algo_full} - rrjedhjet jane 0 Gbps. Shtyp SPACE."
        if kind == "done": return f"PERFUNDOI! Throughput Maksimal ({self.algo_label}) = {self.max_flow} Gbps"
        if kind in ("dfs", "bfs"): return f"Iter {it}: {self.algo_label} gjeti rruge!" if self.cur["found"] else f"Iter {it}: nuk ka me rruge."
        if kind == "augment":
            u, v = self.cur["bottle_edge"]
            return f"Iter {it}: bottleneck {self.cur['bottleneck']} Gbps te {NAMES[u]}->{NAMES[v]}"
        if kind == "update": return f"Iter {it}: +{self.cur['bottleneck']} Gbps - totali = {self.max_flow}"
        return ""

    def draw_graph(self):
        s, H, pos = self.screen, self.screen.get_height(), self.positions()
        kind = self.cur["type"]
        path_edges = set(map(tuple, self.cur.get("path", [])))

        pygame.draw.rect(s, C["graph"], (12,80,self.graph_w-24,H-128), border_radius=16)
        pygame.draw.rect(s, (6,10,22), (12,4,self.graph_w-24,76), border_radius=10)
        color = C["yellow"] if self.strategy == "dfs" else C["cyan"]
        text(s, f"{self.algo_full.upper()} ({self.algo_label}) - Network Capacity Planning", 22, C["white"], (self.graph_w//2,24), True, True)
        text(s, "Data Center  ->  Core Routers  ->  Distribution Switches  ->  Central Server", 13, color, (self.graph_w//2,50), True)

        for u, v, cap in EDGES:
            col, width = edge_style(u, v, self.cur)
            draw_link(s, *pos[u], *pos[v], self.flow(u, v), cap, col, width, self.tick, (u, v) in path_edges)
        for u, v, cap in EDGES:
            col, _ = edge_style(u, v, self.cur)
            draw_label(s, *pos[u], *pos[v], self.flow(u, v), cap, col)
        active_nodes = set(self.cur.get("path_nodes", []))
        for n, (x, y) in pos.items():
            draw_node(s, n, x, y, n in active_nodes, self.tick)

        if self.cur.get("iteration", 0) and kind not in ("done", "init"):
            label = {"dfs":"DFS", "bfs":"BFS", "augment":"AUGMENT", "update":"UPDATE"}.get(kind, kind.upper())
            rect = pygame.Rect(self.graph_w//2-80, H-115, 160, 28)
            col = {"dfs":C["yellow"], "bfs":C["yellow"], "augment":C["cyan"], "update":C["green"]}.get(kind, C["dim"])
            pygame.draw.rect(s, (8,12,28), rect, border_radius=6)
            pygame.draw.rect(s, col, rect, 2, border_radius=6)
            text(s, f"Iteracioni {self.cur['iteration']} - {label}", 13, col, rect.center, True, True)

        bg = {"dfs":(48,42,8), "bfs":(8,36,50), "augment":(8,48,58), "update":(8,52,18), "done":(8,50,18), "init":(8,18,50)}.get(kind, (8,12,28))
        br = {"dfs":C["yellow"], "bfs":C["cyan"], "augment":C["cyan"], "update":C["green"], "done":C["green"], "init":C["blue"]}.get(kind, C["dim"])
        rect = pygame.Rect(12, H-46, self.graph_w-24, 36)
        pygame.draw.rect(s, bg, rect, border_radius=8)
        pygame.draw.rect(s, br, rect, 1, border_radius=8)
        text(s, self.status(), 13, C["white"], rect.center, True)

    def button(self, key, label, col, rect, active=False):
        self.btns[key] = rect
        hover = rect.collidepoint(self.mouse)
        pygame.draw.rect(self.screen, col if active else C["btn_h"] if hover else C["btn"], rect, border_radius=8)
        pygame.draw.rect(self.screen, C["white"] if active else col, rect, 2, border_radius=8)
        text(self.screen, label, 11, (15,15,25) if active else col, rect.center, True, True)

    def draw_panel(self):
        s, H, px, pw, pad = self.screen, self.screen.get_height(), self.graph_w, self.screen.get_width()-self.graph_w, 12
        pygame.draw.rect(s, (8,13,28), (px,0,pw,H))
        pygame.draw.rect(s, C["panel"], (px+6,6,pw-12,H-12), border_radius=12)

        bw, gap, by = (pw-pad*2-8)//2, 8, 12
        self.button("dfs", "FORD-FULKERSON (DFS)", C["yellow"], pygame.Rect(px+pad,by,bw,36), self.strategy == "dfs")
        self.button("bfs", "EDMONDS-KARP (BFS)", C["cyan"], pygame.Rect(px+pad+bw+gap,by,bw,36), self.strategy == "bfs")

        nav_y = by + 46
        nav_w = 52
        self.button("prev", "<-", C["orange"], pygame.Rect(px+pad, nav_y, nav_w, 32))
        self.button("next", "->", C["green"], pygame.Rect(px+pad+nav_w+8, nav_y, nav_w, 32))

        y = 78
        text(s, "THROUGHPUT TOTAL", 13, C["dim"], (px+pw//2, y), True, True)
        text(s, self.max_flow, 48, C["green"] if self.max_flow else C["dim"], (px+pw//2, y+44), True, True)
        text(s, "Gbps", 12, C["dim"], (px+pw//2, y+72), True)
        bar = pygame.Rect(px+pad, y+90, pw-pad*2, 10)
        pygame.draw.rect(s, (20,35,60), bar, border_radius=4)
        pygame.draw.rect(s, C["green"], (px+pad, y+90, int((pw-pad*2)*self.max_flow/27), 10), border_radius=4)
        pygame.draw.rect(s, (40,65,100), bar, 1, border_radius=4)
        y += 112
        pygame.draw.line(s, (30,45,75), (px+pad,y), (px+pw-pad,y)); y += 8
        y = self.draw_step_info(px, pw, y, pad)
        y = self.draw_edge_state(px, pw, y, pad)
        self.draw_log(px, pw, H, pad)

    def draw_step_info(self, px, pw, y, pad):
        kind, it = self.cur["type"], self.cur.get("iteration", 0)
        label = {"init":"GATI", "dfs":"DFS", "bfs":"BFS", "augment":"AUGMENT", "update":"UPDATE", "done":"PERFUNDOI"}.get(kind, kind.upper())
        color = {"dfs":C["yellow"], "bfs":C["cyan"], "augment":C["cyan"], "update":C["green"], "done":C["green"], "init":C["blue"]}.get(kind, C["dim"])
        r = pygame.Rect(px+pad, y, 88, 18)
        pygame.draw.rect(self.screen, color, r, border_radius=4)
        text(self.screen, label, 10, (4,6,16), r.center, True, True)
        if it: text(self.screen, f"Iter {it}", 11, C["dim"], (px+pw-pad-8, y+2))
        y += 24

        if kind in ("dfs", "bfs"):
            col = C["green"] if self.cur["found"] else C["orange"]
            text(self.screen, f"{self.algo_label} eksploroi {len(self.cur.get('explored_edges', []))} lidhje", 11, col, (px+pad, y)); y += 16
            text(self.screen, "-> Gjeti rruge!" if self.cur["found"] else "-> Nuk ka me rruge!", 11, col, (px+pad, y), bold=True); y += 18
        elif kind == "augment":
            u, v = self.cur["bottle_edge"]
            path = " -> ".join(NAMES[n] for n in self.cur["path_nodes"])
            text(self.screen, f"Rruga {self.algo_label}:", 11, C["dim"], (px+pad, y)); y += 14
            text(self.screen, path, 11, C["cyan"], (px+pad+8, y)); y += 16
            text(self.screen, f"({len(self.cur['path_nodes'])-1} hapa)", 10, C["yellow"] if self.strategy == "dfs" else C["cyan"], (px+pad+8, y)); y += 16
            r = pygame.Rect(px+pad, y, pw-pad*2, 22)
            pygame.draw.rect(self.screen, (50,10,10), r, border_radius=4)
            pygame.draw.rect(self.screen, C["red"], r, 1, border_radius=4)
            text(self.screen, f"Bottleneck: {NAMES[u]}->{NAMES[v]} = {self.cur['bottleneck']} Gbps", 11, C["red"], r.center, True, True); y += 26
        elif kind == "update":
            r = pygame.Rect(px+pad, y, pw-pad*2, 22)
            pygame.draw.rect(self.screen, (10,50,18), r, border_radius=4)
            pygame.draw.rect(self.screen, C["green"], r, 1, border_radius=4)
            text(self.screen, f"+{self.cur['bottleneck']} Gbps shtohen - totali: {self.cur['max_flow']}", 11, C["green"], r.center, True, True); y += 28
        elif kind == "done":
            r = pygame.Rect(px+pad, y, pw-pad*2, 50)
            pygame.draw.rect(self.screen, (8,48,18), r, border_radius=6)
            pygame.draw.rect(self.screen, C["green"], r, 2, border_radius=6)
            text(self.screen, f"{self.algo_full.upper()} PERFUNDOI!", 12, C["green"], (px+pw//2,y+12), True, True)
            text(self.screen, f"Max-Flow = {self.cur['max_flow']} Gbps", 11, (180,255,180), (px+pw//2,y+30), True); y += 56
        return y

    def draw_edge_state(self, px, pw, y, pad):
        y += 8
        pygame.draw.line(self.screen, (30,45,75), (px+pad,y), (px+pw-pad,y)); y += 6
        text(self.screen, "GJENDJA E BRINJEVE (Gbps)", 11, C["dim"], (px+pad,y), bold=True); y += 16
        path, bottle = set(map(tuple, self.cur.get("path", []))), self.cur.get("bottle_edge")
        for u, v, cap in EDGES:
            if y > self.screen.get_height()-110: break
            fl = self.flow(u, v)
            col = C["red"] if (u,v)==bottle else C["cyan"] if (u,v) in path else C["green"] if fl else (60,75,100)
            max_w = pw - pad*2 - 65
            pygame.draw.rect(self.screen, (15,22,40), (px+pad,y+2,max_w,8), border_radius=3)
            if fl: pygame.draw.rect(self.screen, col, (px+pad,y+2,int(max_w*fl/cap),8), border_radius=3)
            text(self.screen, f"{NAMES[u]}->{NAMES[v]} {fl}/{cap}", 9, col, (px+pad+max_w+4,y-1))
            y += 13
        return y

    def draw_log(self, px, pw, H, pad):
        y = H - 130
        pygame.draw.line(self.screen, (30,45,75), (px+pad,y), (px+pw-pad,y)); y += 5
        text(self.screen, "LOG", 11, C["dim"], (px+pad,y), bold=True)
        text(self.screen, f"Hapi {self.step_idx}/{len(self.mf.steps)}", 10, C["dim"], (px+pw-pad-8,y)); y += 14
        limit = max(26, (pw-pad*2-8)//7)
        for msg, col in self.logs[-6:]:
            text(self.screen, msg if len(msg) <= limit else msg[:limit-2] + "..", 10, col, (px+pad+3,y+1))
            y += 14
            if y > H - 22: break
        for line in ["SPACE: luaj/pauzo  |  S/->: hap  |  R: rinicio", "Klik buton: ndrysho algoritmin  |  ESC: dil"]:
            text(self.screen, line, 10, (65,85,130), (px+pad+2,y)); y += 13

    def draw(self):
        self.screen.fill(C["bg"])
        self.draw_graph()
        self.draw_panel()
        pygame.display.flip()

    def resize(self, w, h):
        self.graph_w = int(w * .72)

    def toggle_full(self):
        self.full = not self.full
        self.screen = pygame.display.set_mode((0,0), pygame.FULLSCREEN) if self.full else pygame.display.set_mode(self.windowed, pygame.RESIZABLE)
        self.resize(*self.screen.get_size())

    def events(self):
        for e in pygame.event.get():
            if e.type == pygame.QUIT: return False
            if e.type == pygame.VIDEORESIZE and not self.full:
                self.windowed = (e.w, e.h)
                self.screen = pygame.display.set_mode(self.windowed, pygame.RESIZABLE)
                self.resize(e.w, e.h)
            if e.type == pygame.MOUSEMOTION:
                self.mouse = e.pos
            if e.type == pygame.MOUSEBUTTONDOWN and e.button == 1:
                if self.btns.get("dfs", pygame.Rect(0,0,0,0)).collidepoint(e.pos): self.set_strategy("dfs")
                if self.btns.get("bfs", pygame.Rect(0,0,0,0)).collidepoint(e.pos): self.set_strategy("bfs")
                if self.btns.get("next", pygame.Rect(0,0,0,0)).collidepoint(e.pos): self.apply_step()
                if self.btns.get("prev", pygame.Rect(0,0,0,0)).collidepoint(e.pos) and self.step_idx > 1:
                    self.step_idx -= 2
                    self.apply_step()
            if e.type == pygame.KEYDOWN:
                if e.key == pygame.K_ESCAPE: return False
                if e.key == pygame.K_SPACE: self.paused = not self.paused
                if e.key in (pygame.K_s, pygame.K_RIGHT):
                    self.apply_step()
                elif e.key == pygame.K_LEFT and self.step_idx > 1:
                    self.step_idx -= 2
                    self.apply_step()
                if e.key == pygame.K_r: self.reset()
                if e.key in (pygame.K_f, pygame.K_F11): self.toggle_full()
                if e.key == pygame.K_1: self.set_strategy("dfs")
                if e.key == pygame.K_2: self.set_strategy("bfs")
        return True

    def run(self):
        self.resize(*self.windowed)
        while self.events():
            self.tick += 1
            if not self.paused:
                self.counter += 1
                if self.counter >= max(1, 60 // self.speed):
                    self.apply_step()
                    self.counter = 0
            self.draw()
            self.clock.tick(60)
        pygame.quit()


def print_results():
    for strategy in ("dfs", "bfs"):
        algo = "Ford-Fulkerson" if strategy == "dfs" else "Edmonds-Karp"
        label = strategy.upper()
        mf = MaxFlow(strategy)
        done = next(s for s in reversed(mf.steps) if s["type"] == "done")
        snap, total = done["flow_snap"], done["max_flow"]
        print("\n" + "="*70)
        print(f"  {algo} ({label}) - REZULTATET FINALE")
        print("  Network Capacity Planning")
        print("="*70)
        print(f"\n  Throughput Maksimal = {total} Gbps")
        print(f"  Iteracione: {sum(1 for s in mf.steps if s['type'] == 'augment')}")
        print(f"  Hapa total: {len(mf.steps)}")
        print(f"  Strategjia: {label}")
        print(f"\n  Rruget e zgjedhura nga {label}:")
        for s in mf.steps:
            if s["type"] == "augment":
                print(f"  [{s['iteration']}] {' -> '.join(NAMES[n] for n in s['path_nodes'])}  (gjatesia: {len(s['path_nodes'])} nyje)")
                print(f"       Bottleneck: {s['bottleneck']} Gbps")
        print("\n  Rrjedha Final per Lidhje:")
        for u, v, cap in EDGES:
            fl = max(0, snap.get(u, {}).get(v, 0))
            if fl:
                print(f"  {NAMES[u]:<6} -> {NAMES[v]:<6}: {fl:>2}/{cap:<2} Gbps [{'#'*fl + '.'*(cap-fl)}] {int(100*fl/cap)}%")
        print("="*70)


if __name__ == "__main__":
    print_results()
    App().run()


---

## Ushtrimi 4 — Aeroporti Ndërkombëtar: Rrjedha e Pasagjerëve

---

### Skenari Real: Aeroporti Ndërkombëtar TIA

Ti je **menaxher operacional** i aeroportit **TIA International**. Sapo ka përfunduar zgjerimi i terminalit dhe duhet të analizosh nëse infrastruktura e re mund të mbajë ngarkesën maksimale të pasagjerëve gjatë orës së pikut — para nisjes së fluturimeve të mëngjesit.

Çdo pasagjer hyn nga **Terminali Kryesor (T-MAIN)**, kalon nëpër disa pika kontrolli dhe mbërrin te **Zona e Dorëzimit të Bagazheve (BAGGAGE)**. Pyetja operative është: **sa pasagjerë në minutë mund të kalojnë njëkohësisht pa u bllokuar asnjë pikë?**

---

### Çfarë ndodh në secilën pikë

#### T-MAIN — Terminali Kryesor (Burimi)
Të gjithë pasagjerët hyjnë nga porta kryesore e terminalit. Dy korridore dalëse i ndajnë:
- **Gate-A** (kapacitet **18 p/min**) — shërbejmë fluturimet nga veriu, linja Schengen
- **Gate-B** (kapacitet **14 p/min**) — shërbejmë fluturimet nga jugu, linja ndërkombëtare

#### Gate-A dhe Gate-B — Portat e Nisjes
Këtu bëhet **skanimi i parë i biletës dhe pasaportës**. Çdo portë ka numër të kufizuar sportelesh të hapura:
- **Gate-A → Korr-N** (10 p/min) — pasagjerë me fluturime për vendet e BE-së
- **Gate-A → Korr-S** (8 p/min) — pasagjerë me fluturime jashtë BE-së
- **Gate-B → Korr-S** (9 p/min) — hyrje alternative në korridorin qendror
- **Gate-B → Korr-W** (11 p/min) — pasagjerë me nisje perëndimore

#### Korridoret — Rrugët Ndërmjetëse
Pas portave, pasagjerët hyjnë në tre korridore të brendshme të terminalit:

- **Korr-N (Korridori Verior)** — rezervuar për **pasagjerët e Unionit Europian me fluturime Schengen**. Ky koridor ka marrëveshje të posaçme me autoritetin e sigurisë: pasagjerët që udhëtojnë brenda zonës Schengen i nënshtrohen vetëm **skanimit të bagazheve në hyrje të avionit**, duke anashkaluar kontrollin e plotë të sigurisë. Prandaj **Korr-N shkon drejtpërdrejt te BAGGAGE** (8 p/min), pa kaluar fare nëpër Siguria. Kjo është praktikë standarde në shumë aeroporte europiane për fluturimet e brendshme.

- **Korr-S (Korridori Qendror)** — korridori më i ngarkuar, merr pasagjerë nga të dyja portat. Pasagjerët ndahen:
  - Një pjesë drejtohet te **Siguria** (13 p/min) — kontrolli i plotë
  - Pjesa tjetër ka dalje direkte te **BAGGAGE** (10 p/min) — pasagjerë me biletë prioritare që kanë bërë check-in online me skanim të avancuar

- **Korr-W (Korridori Perëndimor)** — korridori më i qetë, vetëm nga Gate-B. Çon te **Siguria** (6 p/min) — kapacitet i ulët sepse ka vetëm dy kabina kontrolli operative.

#### Siguria — Kontrolli i Sigurisë
Pika kritike ku bashkohen rrjedhat nga Korr-S dhe Korr-W. Këtu bëhet:
- Skanimi me rreze X i bagazheve kabine
- Kalimi nëpër portën metalike
- Kontrolli manual nëse aktivizohet alarmi

Kapaciteti dalës: **15 p/min** drejt BAGGAGE — kjo është shpesh **ngushtica (bottleneck)** e të gjithë sistemit.

#### BAGGAGE — Zona e Dorëzimit të Bagazheve (Derdhja)
Destinacioni final ku pasagjerët dorëzojnë bagazhet dhe marrin numrin e vendosjes. Merr rrjedhje nga tre burime:
- Nga **Korr-N** drejtpërdrejt (8 p/min) — pasagjerë BE, skanim i shpejtë
- Nga **Korr-S** drejtpërdrejt (10 p/min) — pasagjerë me prioritet
- Nga **Siguria** (15 p/min) — pasagjerë pas kontrollit të plotë

---

### Pyetja operative
**Sa pasagjerë maksimalisht mund të kalojnë nga T-MAIN te BAGGAGE në minutë?**

Algoritmi do të gjejë **rrugët rritëse** dhe do të llogarisë **rrjedhën maksimale** — kjo do të tregojë edhe **ngushticen** e sistemit, pra pikën ku duhet të investohet për të rritur kapacitetin e aeroportit.

---

### Kontrollet
| Tastë / Buton | Funksioni |
|:---:|---|
| **◀ PREV / NEXT ▶** | Hap pas hapi — para ose mbrapa |
| **← →** | E njëjta gjë me tastet e shigjetave |
| **SPACE / S** | Hap tjetër |
| **1** | Kalo te Edmonds-Karp (BFS) |
| **2** | Kalo te Ford-Fulkerson (DFS) |
| **R** | Rinicio | **ESC** | Dil |


In [ ]:
import pygame, math, sys
from collections import deque, defaultdict

pygame.init()

# ── Ngjyra & fonti ──────────────────────────────────────────────────────────
C = {
    "bg":        (8, 14, 28),   "graph":  (10, 18, 35),  "panel":  (15, 24, 46),
    "white":     (230,245,255), "dim":    (100,130,170),  "blue":   (60,180,255),
    "cyan":      (40,220,255),  "yellow": (240,190, 40),  "red":    (255, 80, 55),
    "green":     (60,220,120),  "orange": (255,160, 60),  "purple": (170,120,255),
    "pipe":      (70,110,150),  "pipe_dk":(18, 30, 50),   "water":  (45,185,240),
    "btn":       (22, 34, 60),  "btn_h":  (32, 48, 80),
}
FONTS = {}
def fnt(sz, bold=False):
    k = (sz, bold)
    if k not in FONTS: FONTS[k] = pygame.font.SysFont("Segoe UI", sz, bold=bold)
    return FONTS[k]
def txt(s, msg, sz, col, pos, center=False, bold=False):
    img = fnt(sz, bold).render(str(msg), True, col)
    r = img.get_rect(center=pos) if center else img.get_rect(topleft=pos)
    s.blit(img, r); return r
def glow(s, x, y, col, rad):
    for r in range(rad, 0, -10):
        sf = pygame.Surface((2*r, 2*r), pygame.SRCALPHA)
        pygame.draw.circle(sf, (*col, int(80*r/rad)), (r,r), r)
        s.blit(sf, (x-r, y-r))

# ── Graf: Aeroporti nderkombëtar ─────────────────────────────────────────────
#   Burim: terminali kryesor (T-MAIN)
#   Derdhje: zona e dorëzimit te bagazheve (BAGGAGE)
#   Nyjet ndërmjetëse: porta, korridore, kontrolli i sigurisë

SOURCE, SINK = 0, 7
EDGES = [
    (0,1,18), (0,2,14),          # T-MAIN → Gate-A, Gate-B
    (1,3,10), (1,4, 8),          # Gate-A → Korr-N, Korr-S
    (2,4, 9), (2,5,11),          # Gate-B → Korr-S, Korr-W
    (3,6, 7), (4,6,13), (5,6,6), # Korridoret → Siguria
    (3,7, 8), (4,7,10), (6,7,15) # Korridoret/Siguria → BAGGAGE
]
NAMES = {0:"T-MAIN",1:"Gate-A",2:"Gate-B",3:"Korr-N",4:"Korr-S",5:"Korr-W",6:"Siguria",7:"BAGGAGE"}

# ── Algoritmet: Ford-Fulkerson (DFS) dhe Edmonds-Karp (BFS) ─────────────────
class MaxFlow:
    def __init__(self, strategy="bfs"):
        self.strategy = strategy
        self.cap  = defaultdict(lambda: defaultdict(int))
        self.adj  = defaultdict(set)
        self.flow = defaultdict(lambda: defaultdict(int))
        self.steps = []
        for u,v,c in EDGES:
            self.cap[u][v] += c; self.adj[u].add(v); self.adj[v].add(u)
        self._run()

    def _snap(self):  return {u: dict(self.flow[u]) for u in self.flow}
    def _res(self,u,v): return self.cap[u][v] - self.flow[u][v]

    def _bfs(self):
        par, seen, q, exp = {SOURCE:None}, {SOURCE}, deque([SOURCE]), []
        while q:
            u = q.popleft()
            for v in sorted(self.adj[u]):
                if v not in seen and self._res(u,v) > 0:
                    seen.add(v); par[v]=u; exp.append((u,v))
                    if v == SINK:
                        path, cur = [], SINK
                        while cur != SOURCE: prev=par[cur]; path.append((prev,cur)); cur=prev
                        return list(reversed(path)), exp
                    q.append(v)
        return None, exp

    def _dfs(self):
        seen, exp = {SOURCE}, []
        def go(u, path):
            if u == SINK: return path
            for v in sorted(self.adj[u]):
                if v not in seen and self._res(u,v) > 0:
                    seen.add(v); exp.append((u,v))
                    r = go(v, path+[(u,v)])
                    if r is not None: return r
            return None
        path = go(SOURCE, [])
        return path, exp

    def _step(self, kind, **kw):
        kw.update(type=kind, snap=self._snap())
        self.steps.append(kw)

    def _run(self):
        total = 0; it = 0
        self._step("init", total=total)
        while True:
            it += 1
            path, exp = self._bfs() if self.strategy=="bfs" else self._dfs()
            self._step(self.strategy, it=it, exp=exp, found=path is not None, total=total)
            if path is None:
                self._step("done", total=total); break
            bn = min(self._res(u,v) for u,v in path)
            be = min(path, key=lambda e: self._res(*e))
            nodes = [SOURCE] + [v for _,v in path]
            self._step("augment", it=it, path=path, nodes=nodes, be=be, bn=bn, total=total)
            for u,v in path: self.flow[u][v]+=bn; self.flow[v][u]-=bn
            total += bn
            self._step("update", it=it, path=path, be=be, bn=bn, total=total)


# ── Vizatimi i nyjeve ────────────────────────────────────────────────────────
def draw_terminal(s, x, y, active):
    if active: glow(s, x, y, C["cyan"], 85)
    pygame.draw.rect(s, (30,50,80), (x-54,y-36,108,72), border_radius=8)
    pygame.draw.rect(s, (100,160,210),(x-54,y-36,108,72), 2, border_radius=8)
    for i,ox in enumerate([-30,0,30]):
        pygame.draw.rect(s,(16,28,50),(x+ox-11,y-26,22,52),border_radius=3)
        lc = C["cyan"] if active and (pygame.time.get_ticks()//300+i)%3==0 else (50,80,110)
        for j in range(4): pygame.draw.circle(s,lc,(x+ox,y-16+j*14),3)
    txt(s,"T-MAIN",14,C["white"],(x,y+50),True,True)
    txt(s,"BURIM",10,C["blue"],(x,y+66),True)

def draw_gate(s, x, y, label, active):
    if active: glow(s, x, y, C["yellow"], 70)
    pygame.draw.rect(s,(40,55,80),(x-36,y-30,72,60),border_radius=6)
    pygame.draw.rect(s,(160,190,220),(x-36,y-30,72,60),2,border_radius=6)
    pygame.draw.rect(s,(20,32,50),(x-24,y-18,48,36),border_radius=3)
    lc = C["green"] if active else (60,80,110)
    pygame.draw.circle(s,lc,(x,y),10)
    txt(s,label,13,C["white"],(x,y+44),True,True)
    txt(s,"Porta",9,C["dim"],(x,y+58),True)

def draw_corridor(s, x, y, label, active):
    if active: glow(s, x, y, C["purple"], 65)
    pygame.draw.rect(s,(35,48,70),(x-34,y-28,68,56),border_radius=5)
    pygame.draw.rect(s,(130,155,195),(x-34,y-28,68,56),2,border_radius=5)
    for i in range(3):
        yy = y-14+i*14
        pygame.draw.rect(s,(18,28,45),(x-22,yy,44,10),border_radius=2)
        if active and (pygame.time.get_ticks()//200+i*2)%5<3:
            pygame.draw.rect(s,C["purple"],(x-20,yy+2,40,6),border_radius=1)
    txt(s,label,13,C["white"],(x,y+42),True,True)
    txt(s,"Korr.",9,C["dim"],(x,y+56),True)

def draw_security(s, x, y, active):
    if active: glow(s, x, y, C["red"], 75)
    pygame.draw.rect(s,(60,40,40),(x-40,y-34,80,68),border_radius=6)
    pygame.draw.rect(s,(200,100,90),(x-40,y-34,80,68),2,border_radius=6)
    pygame.draw.rect(s,(30,20,20),(x-26,y-22,52,44),border_radius=3)
    lc = C["red"] if active else (90,50,50)
    pygame.draw.line(s,lc,(x-16,y),(x+16,y),4)
    pygame.draw.line(s,lc,(x,y-16),(x,y+16),4)
    txt(s,"Siguria",13,C["red"],(x,y+46),True,True)

def draw_baggage(s, x, y, active):
    if active: glow(s, x, y, C["green"], 85)
    pygame.draw.ellipse(s,(60,80,60),(x-46,y-34,92,68))
    pygame.draw.ellipse(s,(130,200,130),(x-46,y-34,92,68),3)
    t = pygame.time.get_ticks()*0.002
    for i in range(5):
        a = t + i*2*math.pi/5
        bx,by = int(x+math.cos(a)*26), int(y+math.sin(a)*18)
        pygame.draw.rect(s,(180,220,180),(bx-7,by-5,14,10),border_radius=3)
    txt(s,"BAGGAGE",14,C["white"],(x,y+50),True,True)
    txt(s,"DERDHJE",9,C["green"],(x,y+66),True)

def draw_node(s, n, x, y, active):
    if n==SOURCE:    draw_terminal(s,x,y,active)
    elif n==1:       draw_gate(s,x,y,"Gate-A",active)
    elif n==2:       draw_gate(s,x,y,"Gate-B",active)
    elif n in(3,4,5):draw_corridor(s,x,y,NAMES[n],active)
    elif n==6:       draw_security(s,x,y,active)
    else:            draw_baggage(s,x,y,active)

# ── Vizatimi i harqeve ───────────────────────────────────────────────────────
def draw_edge(s, p1, p2, fl, cap, color, tick):
    x1,y1=p1; x2,y2=p2
    dx,dy=x2-x1,y2-y1; L=math.hypot(dx,dy) or 1
    w = max(7, 6+int(cap*.35))
    pygame.draw.line(s,C["pipe_dk"],p1,p2,w+8)
    pygame.draw.line(s,color,p1,p2,w)
    pygame.draw.line(s,(185,215,245),p1,p2,max(1,w//6))
    if fl>0:
        pygame.draw.line(s,C["water"],p1,p2,max(4,w//2))
        off=(tick*.013)%1
        for i in range(max(3,int(fl*.4))):
            t=(off+i/max(3,int(fl*.4)))%1
            px,py=int(x1+dx*t),int(y1+dy*t)
            pygame.draw.circle(s,C["water"],(px,py),4)
            pygame.draw.circle(s,(200,240,255),(px,py),2)
    # shigjeta drejtimi
    for t in (.30,.65):
        ax,ay=x1+dx*t,y1+dy*t
        bx,by=x1+dx*(t+.06),y1+dy*(t+.06)
        angle=math.atan2(dy,dx)
        pts=[(int(bx),int(by)),
             (int(bx-11*math.cos(angle-.45)),int(by-11*math.sin(angle-.45))),
             (int(bx-11*math.cos(angle+.45)),int(by-11*math.sin(angle+.45)))]
        pygame.draw.polygon(s,(255,255,255),pts)
        pygame.draw.polygon(s,(5,8,18),pts,1)

def draw_label(s, p1, p2, fl, cap, color):
    x1,y1=p1; x2,y2=p2
    dx,dy=x2-x1,y2-y1; L=math.hypot(dx,dy) or 1
    mx,my=x1+dx*.45,y1+dy*.45
    nx,ny=-dy/L,dx/L
    lx,ly=int(mx+nx*30),int(my+ny*30)
    msg=f"{fl}/{cap}"
    w=fnt(13,True).size(msg)[0]+14
    r=pygame.Rect(lx-w//2,ly-11,w,22)
    pygame.draw.line(s,(80,110,150),(int(mx),int(my)),(lx,ly),1)
    pygame.draw.rect(s,(2,5,14),r,border_radius=4)
    pygame.draw.rect(s,color,r,1,border_radius=4)
    txt(s,msg,12,C["white"],r.center,True,True)

# ── Pozicionet e nyjeve ──────────────────────────────────────────────────────
def node_pos(gw, gh):
    top,bot=90,gh-70; sp=bot-top
    return {
        0:(int(gw*.08), int(top+sp*.50)),   # T-MAIN
        1:(int(gw*.30), int(top+sp*.25)),   # Gate-A
        2:(int(gw*.30), int(top+sp*.75)),   # Gate-B
        3:(int(gw*.55), int(top+sp*.12)),   # Korr-N
        4:(int(gw*.55), int(top+sp*.68)),   # Korr-S
        5:(int(gw*.55), int(top+sp*.88)),   # Korr-W
        6:(int(gw*.76), int(top+sp*.50)),   # Siguria
        7:(int(gw*.93), int(top+sp*.50)),   # BAGGAGE
    }

# ── Aplikacioni kryesor ──────────────────────────────────────────────────────
class App:
    def __init__(self):
        self.screen = pygame.display.set_mode((1440,860), pygame.RESIZABLE)
        self.clock  = pygame.time.Clock()
        self.tick   = 0; self.paused = True; self.full = False
        self.strategy = "bfs"; self.btns = {}; self.mouse = (0,0)
        self.reset()

    def _gw(self):  return int(self.screen.get_width()*.72)
    def _gh(self):  return self.screen.get_height()

    def reset(self):
        pygame.display.set_caption(f"{'Edmonds-Karp' if self.strategy=='bfs' else 'Ford-Fulkerson'} — Aeroporti")
        self.mf      = MaxFlow(self.strategy)
        self.idx     = 0; self.mxflow = 0
        self.cur     = dict(type="init",snap={},path=[],exp=[],be=None,bn=0,nodes=[],total=0,it=0)
        self.logs    = [("Gati. ← → ose SPACE/S per navigim.", C["blue"])]

    def _flow(self,u,v): return max(0, self.cur["snap"].get(u,{}).get(v,0))

    def _apply(self, delta=1):
        self.idx = max(0, min(len(self.mf.steps)-1, self.idx+delta))
        st = self.mf.steps[self.idx]
        self.cur = {
            "type": st["type"], "snap": st["snap"],
            "path": st.get("path",[]), "exp": st.get("exp",[]),
            "be": st.get("be"), "bn": st.get("bn",0),
            "nodes": st.get("nodes",[]), "total": st.get("total",0),
            "it": st.get("it",0),
        }
        self.mxflow = self.cur["total"]
        self._log(st)

    def _log(self, st):
        kind = st["type"]
        algo = "EK/BFS" if self.strategy=="bfs" else "FF/DFS"
        if kind in("bfs","dfs"):
            msg = f"[{st['it']}] {algo}: {'gjeti rruge' if st['found'] else 'nuk ka me rruge'}"
            self.logs.append((msg, C["yellow"] if st["found"] else C["orange"]))
        elif kind=="augment":
            self.logs.append((f"[{st['it']}] Rruge: {' → '.join(NAMES[n] for n in st['nodes'])}", C["cyan"]))
            self.logs.append((f"     Ngushtice: {NAMES[st['be'][0]]}→{NAMES[st['be'][1]]} = {st['bn']}", C["red"]))
        elif kind=="update":
            self.logs.append((f"[{st['it']}] +{st['bn']} → totali = {st['total']}", C["green"]))
        elif kind=="done":
            self.logs.append((f"PERFUNDOI — Rrjedha max = {st['total']}", C["green"]))
        self.logs = self.logs[-20:]

    def _edge_col(self, u, v):
        path = set(map(tuple,self.cur["path"])); exp=set(map(tuple,self.cur["exp"]))
        be = self.cur["be"]
        if be and (u,v)==tuple(be): return C["red"]
        if (u,v) in path:  return C["cyan"]
        if (u,v) in exp:   return C["yellow"]
        return C["pipe"]

    def _btn(self, key, label, col, rect, active=False):
        self.btns[key]=rect
        hover=rect.collidepoint(self.mouse)
        pygame.draw.rect(self.screen,col if active else C["btn_h"] if hover else C["btn"],rect,border_radius=6)
        pygame.draw.rect(self.screen,C["white"] if active else col,rect,2,border_radius=6)
        txt(self.screen,label,11,(10,10,20) if active else col,rect.center,True,True)

    def _draw_graph(self):
        s=self.screen; gw,gh=self._gw(),self._gh()
        pos=node_pos(gw,gh)
        active_n=set(self.cur["nodes"])
        pygame.draw.rect(s,C["graph"],(12,82,gw-24,gh-130),border_radius=14)
        pygame.draw.rect(s,(6,10,22),(12,4,gw-24,78),border_radius=10)
        algo_name = "EDMONDS-KARP (BFS)" if self.strategy=="bfs" else "FORD-FULKERSON (DFS)"
        txt(s,f"{algo_name} — Aeroporti Ndërkombëtar",22,C["white"],(gw//2,24),True,True)
        txt(s,"T-MAIN → Porta → Korridore → Siguria → BAGGAGE",13,
            C["cyan"] if self.strategy=="bfs" else C["yellow"],(gw//2,52),True)
        for u,v,cap in EDGES:
            fl = self._flow(u,v); col=self._edge_col(u,v)
            draw_edge(s,pos[u],pos[v],fl,cap,col,self.tick)
        for u,v,cap in EDGES:
            fl=self._flow(u,v); col=self._edge_col(u,v)
            draw_label(s,pos[u],pos[v],fl,cap,col)
        for n,(x,y) in pos.items():
            draw_node(s,n,x,y,n in active_n)
        # shirit statusi
        kind=self.cur["type"]
        msgs={
            "init": "Shtyp SPACE ose → per te filluar hap pas hapi.",
            "bfs":  f"Iter {self.cur['it']}: BFS kërkon rrugë rritëse.",
            "dfs":  f"Iter {self.cur['it']}: DFS kërkon rrugë rritëse.",
            "augment": f"Iter {self.cur['it']}: rrugë e gjetur — ngushtice {self.cur['bn']}.",
            "update":  f"Iter {self.cur['it']}: +{self.cur['bn']} rrjedhë u shtua. Totali = {self.mxflow}.",
            "done": f"Algoritmi përfundoi. Rrjedha maksimale = {self.mxflow}.",
        }
        bgs={"bfs":(8,36,50),"dfs":(48,42,8),"augment":(8,48,58),"update":(8,52,18),"done":(8,50,18),"init":(8,18,50)}
        r=pygame.Rect(12,gh-46,gw-24,36)
        pygame.draw.rect(s,bgs.get(kind,(8,12,28)),r,border_radius=8)
        pygame.draw.rect(s,C["cyan"] if self.strategy=="bfs" else C["yellow"],r,1,border_radius=8)
        txt(s,msgs.get(kind,""),13,C["white"],r.center,True)

    def _draw_panel(self):
        s=self.screen; gw,gh=self._gw(),self._gh()
        px=gw; pw=s.get_width()-gw; pad=12
        pygame.draw.rect(s,(8,13,28),(px,0,pw,gh))
        pygame.draw.rect(s,C["panel"],(px+6,6,pw-12,gh-12),border_radius=12)
        bw=(pw-pad*2-8)//2; y=14
        self._btn("bfs","EDMONDS-KARP",C["cyan"],pygame.Rect(px+pad,y,bw,34),self.strategy=="bfs")
        self._btn("dfs","FORD-FULKERSON",C["yellow"],pygame.Rect(px+pad+bw+8,y,bw,34),self.strategy=="dfs")
        y+=44
        # shigjetat prev/next
        arw=60
        self._btn("prev","◀ PREV",C["orange"],pygame.Rect(px+pad,y,arw,30))
        self._btn("next","NEXT ▶",C["green"],pygame.Rect(px+pad+arw+8,y,arw,30))
        txt(s,f"Hapi {self.idx+1}/{len(self.mf.steps)}",11,C["dim"],(px+pad+arw*2+24,y+8))
        y+=44
        # rrjedha
        txt(s,"RRJEDHA MAKSIMALE",12,C["dim"],(px+pw//2,y),True,True); y+=20
        txt(s,str(self.mxflow),48,C["green"] if self.mxflow else C["dim"],(px+pw//2,y+22),True,True)
        txt(s,"pasagjere / min",11,C["dim"],(px+pw//2,y+50),True); y+=68
        bar=pygame.Rect(px+pad,y,pw-pad*2,8)
        max_cap=32
        pygame.draw.rect(s,(20,35,60),bar,border_radius=3)
        if self.mxflow: pygame.draw.rect(s,C["green"],(px+pad,y,int((pw-pad*2)*self.mxflow/max_cap),8),border_radius=3)
        pygame.draw.rect(s,(40,65,100),bar,1,border_radius=3)
        y+=20; pygame.draw.line(s,(30,45,75),(px+pad,y),(px+pw-pad,y)); y+=8
        # info hapit aktual
        kind=self.cur["type"]
        col_map={"bfs":C["cyan"],"dfs":C["yellow"],"augment":C["cyan"],"update":C["green"],"done":C["green"],"init":C["blue"]}
        col=col_map.get(kind,C["dim"])
        label={"init":"GATI","bfs":"BFS","dfs":"DFS","augment":"AUGMENT","update":"UPDATE","done":"DONE"}.get(kind,kind.upper())
        r=pygame.Rect(px+pad,y,80,18); pygame.draw.rect(s,col,r,border_radius=3)
        txt(s,label,10,(4,6,16),r.center,True,True)
        if self.cur["it"]: txt(s,f"Iter {self.cur['it']}",10,C["dim"],(px+pw-pad-50,y+2))
        y+=24
        if kind in("bfs","dfs"):
            exp_cnt=len(self.cur["exp"])
            found=any(e[1]==SINK for e in self.cur["exp"])
            txt(s,f"Eksploruar: {exp_cnt} harqe",11,C["dim"],(px+pad,y)); y+=16
            txt(s,"→ Gjeti rrugë!" if found else "→ Nuk ka rrugë.",11,C["green"] if found else C["orange"],(px+pad,y),bold=True); y+=18
        elif kind=="augment":
            path_str=" → ".join(NAMES[n] for n in self.cur["nodes"])
            txt(s,"Rruga:",11,C["dim"],(px+pad,y)); y+=14
            # ndaj rrugën nëse është shumë e gjatë
            words=path_str.split(" → "); mid=len(words)//2
            txt(s," → ".join(words[:mid+1]),10,C["cyan"],(px+pad+6,y)); y+=14
            if mid<len(words)-1: txt(s,"  → "+" → ".join(words[mid+1:]),10,C["cyan"],(px+pad+6,y)); y+=14
            be=self.cur["be"]
            if be:
                r=pygame.Rect(px+pad,y,pw-pad*2,22)
                pygame.draw.rect(s,(50,10,10),r,border_radius=4); pygame.draw.rect(s,C["red"],r,1,border_radius=4)
                txt(s,f"Ngushtice: {NAMES[be[0]]}→{NAMES[be[1]]} = {self.cur['bn']}",11,C["red"],r.center,True,True); y+=26
        elif kind=="update":
            r=pygame.Rect(px+pad,y,pw-pad*2,22)
            pygame.draw.rect(s,(10,50,18),r,border_radius=4); pygame.draw.rect(s,C["green"],r,1,border_radius=4)
            txt(s,f"+{self.cur['bn']} u shtua — totali: {self.cur['total']}",11,C["green"],r.center,True,True); y+=28
        elif kind=="done":
            r=pygame.Rect(px+pad,y,pw-pad*2,50)
            pygame.draw.rect(s,(8,48,18),r,border_radius=6); pygame.draw.rect(s,C["green"],r,2,border_radius=6)
            algo_nm="EDMONDS-KARP" if self.strategy=="bfs" else "FORD-FULKERSON"
            txt(s,f"{algo_nm} PERFUNDOI!",12,C["green"],(px+pw//2,y+12),True,True)
            txt(s,f"Rrjedha max = {self.cur['total']} pasagjere/min",11,(180,255,180),(px+pw//2,y+30),True); y+=56
        y+=8; pygame.draw.line(s,(30,45,75),(px+pad,y),(px+pw-pad,y)); y+=6
        # lista harqe
        txt(s,"HARQET (rrjedhë/kap)",11,C["dim"],(px+pad,y),bold=True); y+=16
        path_set=set(map(tuple,self.cur["path"])); be=self.cur["be"]
        for u,v,cap in EDGES:
            if y>gh-110: break
            fl=self._flow(u,v)
            ec=C["red"] if be and (u,v)==tuple(be) else C["cyan"] if (u,v) in path_set else C["green"] if fl else (55,70,95)
            mw=pw-pad*2-60
            pygame.draw.rect(s,(15,22,40),(px+pad,y+2,mw,7),border_radius=2)
            if fl: pygame.draw.rect(s,ec,(px+pad,y+2,int(mw*fl/cap),7),border_radius=2)
            txt(s,f"{NAMES[u][:5]}→{NAMES[v][:5]} {fl}/{cap}",9,ec,(px+pad+mw+4,y-1))
            y+=13
        # log
        pygame.draw.line(s,(30,45,75),(px+pad,y),(px+pw-pad,y)); y+=4
        txt(s,"LOG",10,C["dim"],(px+pad,y),bold=True); y+=14
        for msg,col in self.logs[-6:]:
            txt(s,msg[:40],10,col,(px+pad,y)); y+=13
            if y>gh-30: break
        # kontrolle
        for line in ["← → : hap prev/next  |  SPACE/S : next","1: EK-BFS  2: FF-DFS  |  R: reset  ESC: dil"]:
            txt(s,line,10,(60,80,125),(px+pad,gh-28+(["← → : hap prev/next  |  SPACE/S : next","1: EK-BFS  2: FF-DFS  |  R: reset  ESC: dil"].index(line))*13))

    def events(self):
        for e in pygame.event.get():
            if e.type==pygame.QUIT: return False
            if e.type==pygame.VIDEORESIZE and not self.full:
                self.screen=pygame.display.set_mode((e.w,e.h),pygame.RESIZABLE)
            if e.type==pygame.MOUSEMOTION: self.mouse=e.pos
            if e.type==pygame.MOUSEBUTTONDOWN and e.button==1:
                if self.btns.get("bfs",pygame.Rect(0,0,0,0)).collidepoint(e.pos) and self.strategy!="bfs":
                    self.strategy="bfs"; self.reset()
                elif self.btns.get("dfs",pygame.Rect(0,0,0,0)).collidepoint(e.pos) and self.strategy!="dfs":
                    self.strategy="dfs"; self.reset()
                elif self.btns.get("next",pygame.Rect(0,0,0,0)).collidepoint(e.pos): self._apply(+1)
                elif self.btns.get("prev",pygame.Rect(0,0,0,0)).collidepoint(e.pos): self._apply(-1)
            if e.type==pygame.KEYDOWN:
                if e.key==pygame.K_ESCAPE: return False
                if e.key in(pygame.K_RIGHT,pygame.K_s,pygame.K_SPACE): self._apply(+1)
                if e.key==pygame.K_LEFT: self._apply(-1)
                if e.key==pygame.K_r: self.reset()
                if e.key==pygame.K_1 and self.strategy!="bfs": self.strategy="bfs"; self.reset()
                if e.key==pygame.K_2 and self.strategy!="dfs": self.strategy="dfs"; self.reset()
                if e.key in(pygame.K_f,pygame.K_F11):
                    self.full=not self.full
                    self.screen=pygame.display.set_mode((0,0),pygame.FULLSCREEN) if self.full else pygame.display.set_mode((1440,860),pygame.RESIZABLE)
        return True

    def run(self):
        while self.events():
            self.tick+=1
            if not self.paused:
                self.tick%=3600
            self.screen.fill(C["bg"])
            self._draw_graph()
            self._draw_panel()
            pygame.display.flip()
            self.clock.tick(60)
        pygame.quit()

if __name__=="__main__":
    App().run()


---

## Loja Finale — "Pipeline Defender"

Një rrjet furnizimi me energji elektrike po transporton energji nga **Centrali** deri te **Qyteti**. Ti je **inxhinieri i mbrojtjes** — duhet të gjesh grupin **minimal** të linjave që, nëse ndërpriten, e bllokojnë plotësisht furnizimin.

Por kujdes — ndërprerja e linjave ka **kosto** (kapaciteti i tyre). Synimi është ta bësh me **kosto sa më të vogël**.

---

### Si luhet
1. Algoritmi llogarit automatikisht **Max-Flow** (furnizimin maksimal)
2. Ti klik mbi **linjat** (brinjët) për t'i ndërprerë
3. Nëse arrin të izolosh Qytetin nga Centrali → prerja është e vlefshme
4. **Skor = kosto e prerjes sate** — sa më e vogël, aq më mirë
5. Kur kosto = Max-Flow → ke gjetur **Min-Cut optimal** → **FITOVE!**

---

### Çfarë mëson kjo lojë
- **Max-Flow = Min-Cut** — provoje vetë me klikime
- Çdo prerje e vlefshme ka kosto ≥ Max-Flow
- Min-Cut është prerja me kosto **saktësisht** = Max-Flow
- Bottleneck i rrjetit = linjat e Min-Cut

> **Kontrollet:** Klik linjë → ndërprit/restauro | `M` trego Min-Cut optimal | `R` reset | `H` ndihmë


In [1]:
import pygame, math
from collections import defaultdict, deque
import random

pygame.init()

W0, H0 = 1440, 860
SOURCE, SINK = 0, 7

# Graf energjetik: Centrali → Nënstacione → Transformatorë → Qyteti
EDGES = [
    (0,1,15),(0,2,12),(0,3,10),
    (1,2, 5),(1,4,10),(1,5, 8),
    (2,3, 4),(2,5, 9),(2,6, 7),
    (3,6,11),(3,7, 6),
    (4,5, 3),(4,7,12),
    (5,6, 6),(5,7, 9),
    (6,7,14),
]
NAMES = {
    0:"Centrali", 1:"NS-A", 2:"NS-B", 3:"NS-C",
    4:"TR-1",    5:"TR-2", 6:"TR-3", 7:"Qyteti"
}

C = {
    "bg":(8,14,28),"graph":(10,18,35),"panel":(15,24,46),
    "white":(230,245,255),"dim":(100,130,170),"blue":(60,180,255),
    "cyan":(40,220,255),"yellow":(240,190,40),"red":(255,80,55),
    "green":(60,220,120),"orange":(255,160,60),"purple":(170,120,255),
    "pipe":(70,110,150),"pipe_dk":(18,30,50),"water":(45,185,240),
    "gold":(255,215,60),"btn":(22,34,60),"btn_h":(32,48,80),
}
FONTS = {}
def fnt(sz, bold=False):
    k=(sz,bold)
    if k not in FONTS: FONTS[k]=pygame.font.SysFont("Segoe UI",sz,bold=bold)
    return FONTS[k]
def txt(s,msg,sz,col,pos,center=False,bold=False):
    img=fnt(sz,bold).render(str(msg),True,col)
    r=img.get_rect(center=pos) if center else img.get_rect(topleft=pos)
    s.blit(img,r); return r
def glow(s,x,y,col,rad):
    for r in range(rad,0,-10):
        sf=pygame.Surface((2*r,2*r),pygame.SRCALPHA)
        pygame.draw.circle(sf,(*col,int(80*r/rad)),(r,r),r)
        s.blit(sf,(x-r,y-r))

# ── BFS max-flow (Edmonds-Karp) ──────────────────────────────────────────────
def compute_max_flow():
    cap=defaultdict(lambda:defaultdict(int))
    adj=defaultdict(set)
    for u,v,c in EDGES: cap[u][v]+=c; adj[u].add(v); adj[v].add(u)
    flow=defaultdict(lambda:defaultdict(int))
    while True:
        par={SOURCE:None}; q=deque([SOURCE])
        while q and SINK not in par:
            u=q.popleft()
            for v in sorted(adj[u]):
                if v not in par and cap[u][v]-flow[u][v]>0:
                    par[v]=u; q.append(v)
        if SINK not in par: break
        path=[]; cur=SINK
        while cur!=SOURCE: prev=par[cur]; path.append((prev,cur)); cur=prev
        b=min(cap[u][v]-flow[u][v] for u,v in path)
        for u,v in path: flow[u][v]+=b; flow[v][u]-=b
    return sum(flow[SOURCE][v] for v in adj[SOURCE])

def compute_min_cut():
    cap=defaultdict(lambda:defaultdict(int))
    adj=defaultdict(set)
    for u,v,c in EDGES: cap[u][v]+=c; adj[u].add(v); adj[v].add(u)
    flow=defaultdict(lambda:defaultdict(int))
    while True:
        par={SOURCE:None}; q=deque([SOURCE])
        while q and SINK not in par:
            u=q.popleft()
            for v in sorted(adj[u]):
                if v not in par and cap[u][v]-flow[u][v]>0:
                    par[v]=u; q.append(v)
        if SINK not in par: break
        path=[]; cur=SINK
        while cur!=SOURCE: prev=par[cur]; path.append((prev,cur)); cur=prev
        b=min(cap[u][v]-flow[u][v] for u,v in path)
        for u,v in path: flow[u][v]+=b; flow[v][u]-=b
    # BFS on residual to find S-side
    seen={SOURCE}; q=deque([SOURCE])
    while q:
        u=q.popleft()
        for v in adj[u]:
            if v not in seen and cap[u][v]-flow[u][v]>0:
                seen.add(v); q.append(v)
    return {(u,v) for u,v,_ in EDGES if u in seen and v not in seen}

# ── partition after user cut ─────────────────────────────────────────────────
def reachable_after_cut(cut_set):
    adj2=defaultdict(set)
    for u,v,_ in EDGES:
        if (u,v) not in cut_set: adj2[u].add(v)
    seen={SOURCE}; q=deque([SOURCE])
    while q:
        u=q.popleft()
        for v in adj2[u]:
            if v not in seen: seen.add(v); q.append(v)
    return seen

# ── pozicionet e nyjeve ───────────────────────────────────────────────────────
def node_pos(gw,gh):
    top,bot=95,gh-75; sp=bot-top
    return {
        0:(int(gw*.07),int(top+sp*.50)),
        1:(int(gw*.26),int(top+sp*.18)),
        2:(int(gw*.28),int(top+sp*.50)),
        3:(int(gw*.26),int(top+sp*.82)),
        4:(int(gw*.54),int(top+sp*.12)),
        5:(int(gw*.52),int(top+sp*.50)),
        6:(int(gw*.54),int(top+sp*.82)),
        7:(int(gw*.85),int(top+sp*.50)),
    }

def dist_seg(px,py,x1,y1,x2,y2):
    dx,dy=x2-x1,y2-y1; l2=dx*dx+dy*dy
    if not l2: return math.hypot(px-x1,py-y1)
    t=max(0,min(1,((px-x1)*dx+(py-y1)*dy)/l2))
    return math.hypot(px-(x1+t*dx),py-(y1+t*dy))

# ── vizatimi i nyjeve ─────────────────────────────────────────────────────────
def draw_central(s,x,y,tick):
    glow(s,x,y,C["yellow"],90)
    pygame.draw.rect(s,(50,45,15),(x-48,y-42,96,84),border_radius=6)
    pygame.draw.rect(s,(220,180,40),(x-48,y-42,96,84),2,border_radius=6)
    for i,ox in enumerate([-24,0,24]):
        pygame.draw.rect(s,(30,28,8),(x+ox-10,y-30,20,60),border_radius=3)
        lc=C["yellow"] if (tick//20+i)%3<2 else (80,70,10)
        for j in range(4): pygame.draw.circle(s,lc,(x+ox,y-20+j*14),4)
    # shenja elektrike
    pts=[(x-8,y-16),(x+2,y-2),(x-4,y-2),(x+8,y+16),(x-2,y+2),(x+4,y+2)]
    pygame.draw.polygon(s,C["yellow"],pts)
    txt(s,"CENTRALI",13,C["yellow"],(x,y+56),True,True)
    txt(s,"BURIM",9,C["dim"],(x,y+70),True)

def draw_substation(s,x,y,label,active,cut):
    col=C["red"] if cut else C["cyan"] if active else C["blue"]
    if active: glow(s,x,y,col,60)
    pygame.draw.rect(s,(25,40,65),(x-34,y-30,68,60),border_radius=5)
    pygame.draw.rect(s,col,(x-34,y-30,68,60),2,border_radius=5)
    pygame.draw.rect(s,(15,25,45),(x-22,y-18,44,36),border_radius=3)
    # bobina transformatori
    for i in range(3):
        pygame.draw.circle(s,col,(x-12+i*12,y-4),6,2)
        pygame.draw.circle(s,col,(x-12+i*12,y+8),6,2)
    txt(s,label,13,C["white"],(x,y+42),True,True)

def draw_transformer(s,x,y,label,active,cut):
    col=C["red"] if cut else C["purple"] if active else (100,80,160)
    if active: glow(s,x,y,col,60)
    pygame.draw.rect(s,(30,22,50),(x-32,y-28,64,56),border_radius=5)
    pygame.draw.rect(s,col,(x-32,y-28,64,56),2,border_radius=5)
    # simboli transformatori
    pygame.draw.circle(s,col,(x,y-8),14,2)
    pygame.draw.circle(s,col,(x,y+8),14,2)
    txt(s,label,13,C["white"],(x,y+40),True,True)

def draw_city(s,x,y,powered,tick):
    col=C["green"] if powered else C["red"]
    glow(s,x,y,col,85)
    # ndertesa
    for i,(ox,h) in enumerate([(-28,50),(-8,65),(12,45),(28,55)]):
        bx=x+ox
        pygame.draw.rect(s,(40,55,40) if powered else (55,30,30),(bx-10,y-h//2,20,h),border_radius=3)
        pygame.draw.rect(s,col,(bx-10,y-h//2,20,h),1,border_radius=3)
        if powered:
            for wy in range(y-h//2+8,y+h//2-4,12):
                lc=C["yellow"] if (tick//25+i+wy)%4<3 else (60,60,10)
                pygame.draw.rect(s,lc,(bx-6,wy,5,5))
                pygame.draw.rect(s,lc,(bx+1,wy,5,5))
    txt(s,"QYTETI",14,C["white"],(x,y+50),True,True)
    txt(s,"FUNDERRJEDHJE" if powered else "E BLLOKUAR",9,col,(x,y+65),True)

def draw_node(s,n,x,y,active,cut,tick):
    if n==SOURCE: draw_central(s,x,y,tick)
    elif n<=3:    draw_substation(s,x,y,NAMES[n],active,cut)
    elif n<=6:    draw_transformer(s,x,y,NAMES[n],active,cut)
    else:         draw_city(s,x,y,not cut,tick)

# ── vizatimi i linjave elektrike ──────────────────────────────────────────────
def draw_line(s,p1,p2,cap,cut,hover,flowing,tick):
    x1,y1=p1; x2,y2=p2
    dx,dy=x2-x1,y2-y1; L=math.hypot(dx,dy) or 1
    w=max(8,6+int(cap*.4))
    pygame.draw.line(s,C["pipe_dk"],p1,p2,w+8)
    if cut:
        mx,my=(x1+x2)/2,(y1+y2)/2
        ux,uy=dx/L,dy/L; gap=20
        a=(mx-ux*gap,my-uy*gap); b=(mx+ux*gap,my+uy*gap)
        pygame.draw.line(s,(50,20,20),p1,p2,w)
        pygame.draw.line(s,(110,40,35),p1,(int(a[0]),int(a[1])),w)
        pygame.draw.line(s,(110,40,35),(int(b[0]),int(b[1])),p2,w)
        pygame.draw.line(s,C["red"],(int(mx-15),int(my-15)),(int(mx+15),int(my+15)),4)
        pygame.draw.line(s,C["red"],(int(mx-15),int(my+15)),(int(mx+15),int(my-15)),4)
        # shkëndija
        for i in range(3):
            t=(tick*.06+i*.33)%1
            sx,sy=int(a[0]+math.sin(tick*.12+i)*4),int(a[1]+8+t*18)
            pygame.draw.circle(s,C["yellow"],(sx,sy),2)
        return
    col=(200,220,80) if hover else C["pipe"]
    pygame.draw.line(s,col,p1,p2,w)
    pygame.draw.line(s,(180,220,255),p1,p2,max(1,w//6))
    if flowing:
        off=(tick*.014)%1
        for i in range(max(3,int(cap*.5))):
            t=(off+i/max(3,int(cap*.5)))%1
            px,py=int(x1+dx*t),int(y1+dy*t)
            pygame.draw.circle(s,C["water"],(px,py),4)
            pygame.draw.circle(s,(200,240,255),(px,py),2)
    # shigjeta drejtimi
    for tf in (.30,.65):
        ax,ay=x1+dx*tf,y1+dy*tf
        bx2,by2=x1+dx*(tf+.06),y1+dy*(tf+.06)
        ang=math.atan2(dy,dx)
        pts=[(int(bx2),int(by2)),
             (int(bx2-12*math.cos(ang-.45)),int(by2-12*math.sin(ang-.45))),
             (int(bx2-12*math.cos(ang+.45)),int(by2-12*math.sin(ang+.45)))]
        pygame.draw.polygon(s,C["white"],pts)
        pygame.draw.polygon(s,(5,8,18),pts,1)

def draw_edge_label(s,p1,p2,cap,cut,is_min):
    x1,y1=p1; x2,y2=p2
    dx,dy=x2-x1,y2-y1; L=math.hypot(dx,dy) or 1
    mx,my=x1+dx*.45,y1+dy*.45
    nx,ny=-dy/L,dx/L
    lx,ly=int(mx+nx*28),int(my+ny*28)
    col=C["red"] if cut else C["gold"] if is_min else C["blue"]
    msg=f"{cap} MW"
    w=fnt(12,True).size(msg)[0]+14
    r=pygame.Rect(lx-w//2,ly-11,w,22)
    pygame.draw.line(s,(80,110,150),(int(mx),int(my)),(lx,ly),1)
    pygame.draw.rect(s,(2,5,14),r,border_radius=4)
    pygame.draw.rect(s,col,r,1,border_radius=4)
    txt(s,msg,12,col,r.center,True,True)

# ── loja kryesore ─────────────────────────────────────────────────────────────
class Game:
    def __init__(self):
        self.screen=pygame.display.set_mode((W0,H0),pygame.RESIZABLE)
        pygame.display.set_caption("Pipeline Defender — Gjej Min-Cut!")
        self.clock=pygame.time.Clock()
        self.tick=0; self.full=False; self.gw=int(W0*.74)
        self.mouse=(0,0); self.hover=None; self.show_min=False; self.show_help=False
        self.mf=compute_max_flow()
        self.optimal_cut=compute_min_cut()
        self.cut=set(); self.btns={}
        self.won=False; self.attempts=0

    def _pos(self): return node_pos(self.gw,self.screen.get_height())

    def _edge_at(self,pos):
        p=self._pos(); best=None; dmin=14
        for u,v,_ in EDGES:
            d=dist_seg(pos[0],pos[1],*p[u],*p[v])
            if d<dmin: best,dmin=(u,v),d
        return best

    def _s_side(self): return reachable_after_cut(self.cut)

    def _cut_cost(self):
        s=self._s_side()
        return sum(c for u,v,c in EDGES if u in s and v not in s)

    def _valid(self): return SINK not in self._s_side()

    def _draw_graph(self):
        s=self.screen; h=self.screen.get_height(); p=self._pos()
        s_side=self._s_side(); valid=self._valid()
        cost=self._cut_cost() if self.cut else 0

        pygame.draw.rect(s,C["graph"],(12,82,self.gw-24,h-130),border_radius=14)
        pygame.draw.rect(s,(6,10,22),(12,4,self.gw-24,78),border_radius=10)
        txt(s,"⚡ PIPELINE DEFENDER — Gjej Min-Cut Optimal!",24,C["yellow"],(self.gw//2,26),True,True)
        txt(s,f"Blloko furnizimin • Max-Flow = {self.mf} MW • Kosto jote: {cost} MW",13,
            C["white"],(self.gw//2,54),True)

        for u,v,cap in EDGES:
            cut=(u,v) in self.cut
            flowing=u in s_side and not cut
            draw_line(s,p[u],p[v],cap,cut,self.hover==(u,v),flowing,self.tick)
        for u,v,cap in EDGES:
            is_min=self.show_min and (u,v) in self.optimal_cut
            draw_edge_label(s,p[u],p[v],cap,(u,v) in self.cut,is_min)
        for n,(x,y) in p.items():
            in_cut=n not in s_side and n!=SOURCE
            draw_node(s,n,x,y,n in s_side,in_cut,self.tick)

        # shirit statusi
        if self.won:
            msg=f"🏆 FITOVE! Min-Cut optimal = {self.mf} MW me {len(self.cut)} linja! Provo me pak linja..."
            bg,br=(8,55,18),C["green"]
        elif not self.cut:
            msg="Klik mbi linjat elektrike për t'i ndërprerë. Gjej grupin minimal!"; bg,br=(8,18,50),C["blue"]
        elif not valid:
            msg=f"Qyteti ende merr energji — prerja nuk është e plotë. ({len(self.cut)} linja)"; bg,br=(48,24,8),C["orange"]
        else:
            msg=f"✓ Prerje e vlefshme! Kosto = {cost} MW. {'= Max-Flow — optimal!' if cost==self.mf else f'> {self.mf} MW — provo të presësh më pak.'}"
            bg,br=((8,52,18),C["green"]) if cost==self.mf else ((8,36,50),C["cyan"])
        r=pygame.Rect(12,h-46,self.gw-24,36)
        pygame.draw.rect(s,bg,r,border_radius=8); pygame.draw.rect(s,br,r,1,border_radius=8)
        txt(s,msg,13,C["white"],r.center,True)

        if self.show_help: self._help_overlay()

    def _btn(self,key,label,col,rect,active=False):
        self.btns[key]=rect; hover=rect.collidepoint(self.mouse)
        pygame.draw.rect(self.screen,col if active else C["btn_h"] if hover else C["btn"],rect,border_radius=6)
        pygame.draw.rect(self.screen,C["white"] if active else col,rect,2,border_radius=6)
        txt(self.screen,label,11,(10,10,20) if active else col,rect.center,True,True)

    def _draw_panel(self):
        s=self.screen; H=self.screen.get_height(); px=self.gw; pw=s.get_width()-self.gw; pad=12
        pygame.draw.rect(s,(8,13,28),(px,0,pw,H))
        pygame.draw.rect(s,C["panel"],(px+6,6,pw-12,H-12),border_radius=12)
        bw=(pw-pad*2-8)//2; y=14
        self._btn("reset","RESET",C["orange"],pygame.Rect(px+pad,y,bw,32))
        self._btn("min","MIN-CUT" if not self.show_min else "FSHIH",C["gold"],pygame.Rect(px+pad+bw+8,y,bw,32),self.show_min)
        y+=44
        txt(s,"MAX-FLOW I RRJETIT",12,C["dim"],(px+pw//2,y),True,True); y+=20
        txt(s,f"{self.mf}",50,C["yellow"],(px+pw//2,y+24),True,True)
        txt(s,"MW",11,C["dim"],(px+pw//2,y+52),True); y+=70
        pygame.draw.line(s,(30,45,75),(px+pad,y),(px+pw-pad,y)); y+=10

        # skor live
        valid=self._valid(); cost=self._cut_cost() if self.cut else 0
        txt(s,"KOSTO E PRERJES SATE",12,C["dim"],(px+pw//2,y),True,True); y+=20
        col_cost=C["green"] if (valid and cost==self.mf) else C["cyan"] if valid else C["orange"] if self.cut else C["dim"]
        txt(s,f"{cost if self.cut else '—'}",42,col_cost,(px+pw//2,y+22),True,True)
        txt(s,"MW",11,C["dim"],(px+pw//2,y+48),True); y+=64

        # progress bar: kosto vs max-flow
        bar=pygame.Rect(px+pad,y,pw-pad*2,10)
        pygame.draw.rect(s,(20,35,60),bar,border_radius=4)
        if self.cut and cost>0:
            fill=min(1,cost/(self.mf*2)); col_bar=C["green"] if cost==self.mf else C["cyan"] if valid else C["red"]
            pygame.draw.rect(s,col_bar,(px+pad,y,int((pw-pad*2)*fill),10),border_radius=4)
        # tick marker at max-flow position
        tick_x=px+pad+int((pw-pad*2)*0.5)
        pygame.draw.line(s,C["yellow"],(tick_x,y-3),(tick_x,y+13),2)
        txt(s,"MIN",9,C["yellow"],(tick_x,y+17),True); y+=28
        pygame.draw.line(s,(30,45,75),(px+pad,y),(px+pw-pad,y)); y+=10

        # linja te nderprera
        txt(s,"LINJAT E NDËRPRERA",12,C["dim"],(px+pad,y),bold=True); y+=18
        if not self.cut:
            txt(s,"(asnjë)",11,C["dim"],(px+pad,y)); y+=16
        for u,v,cap in EDGES:
            if (u,v) in self.cut:
                if y>H-160: break
                is_opt=(u,v) in self.optimal_cut
                col=C["gold"] if is_opt else C["red"]
                mark="★" if is_opt else "✗"
                txt(s,f"{mark} {NAMES[u]}→{NAMES[v]}: {cap} MW",11,col,(px+pad,y)); y+=16
        pygame.draw.line(s,(30,45,75),(px+pad,y),(px+pw-pad,y)); y+=8

        # lista te gjitha brinjeve
        txt(s,"TË GJITHA LINJAT",11,C["dim"],(px+pad,y),bold=True); y+=16
        for u,v,cap in EDGES:
            if y>H-60: break
            cut=(u,v) in self.cut; is_opt=self.show_min and (u,v) in self.optimal_cut
            col=C["red"] if cut else C["gold"] if is_opt else (55,70,95)
            mark="✗" if cut else "◆" if is_opt else "─"
            txt(s,f"{mark} {NAMES[u][:4]}→{NAMES[v][:4]} {cap}MW",9,col,(px+pad,y)); y+=13

        # kontrollet
        for line in ["Klik linjë: ndërprit/restauro","M: min-cut  H: ndihmë  R: reset","ESC: dil"]:
            txt(s,line,10,(60,80,125),(px+pad,H-42+(["Klik linjë: ndërprit/restauro","M: min-cut  H: ndihmë  R: reset","ESC: dil"].index(line))*13))

    def _help_overlay(self):
        s=self.screen; W,H=s.get_size()
        ov=pygame.Surface((W,H),pygame.SRCALPHA); ov.fill((0,0,0,190)); s.blit(ov,(0,0))
        bw,bh=720,440; x,y=(W-bw)//2,(H-bh)//2
        pygame.draw.rect(s,(14,22,44),(x,y,bw,bh),border_radius=12)
        pygame.draw.rect(s,C["yellow"],(x,y,bw,bh),2,border_radius=12)
        txt(s,"⚡ Si funksionon Pipeline Defender",22,C["yellow"],(W//2,y+30),True,True)
        lines=[
            "Rrjeti transporton energji nga CENTRALI te QYTETI.",
            "Secila linjë ka kapacitet (MW) — sa energji mund të kalojë.",
            "",
            "Max-Flow = furnizimi maksimal i mundshëm = "+str(self.mf)+" MW",
            "Min-Cut  = grupi minimal i linjave që e bllokon furnizimin.",
            "",
            "TEOREMA: Max-Flow = Min-Cut (gjithmonë!)",
            "",
            "Si të fitosh:",
            "1. Kliko linja për t'i ndërprerë (X e kuqe = e ndërprerë)",
            "2. Sigurohu që QYTETI nuk merr më energji",
            "3. Kosto e prerjes sate duhet = Max-Flow ("+str(self.mf)+" MW)",
            "4. Sa më pak linja, aq më elegant — por kostoja është ajo që llogaritet!",
            "",
            "Shtyp M për të parë zgjidhjen optimale (★ = linja e min-cut)",
            "",
            "Kliko kudo për ta mbyllur.",
        ]
        for i,line in enumerate(lines):
            col=C["gold"] if line.startswith("TEOREMA") else C["cyan"] if line.startswith("Max") or line.startswith("Min") else C["white"]
            txt(s,line,13,col,(x+30,y+65+i*20))

    def _events(self):
        for e in pygame.event.get():
            if e.type==pygame.QUIT: return False
            if e.type==pygame.VIDEORESIZE and not self.full:
                self.screen=pygame.display.set_mode((e.w,e.h),pygame.RESIZABLE); self.gw=int(e.w*.74)
            if e.type==pygame.MOUSEMOTION:
                self.mouse=e.pos; self.hover=self._edge_at(e.pos) if e.pos[0]<self.gw else None
            if e.type==pygame.MOUSEBUTTONDOWN and e.button==1:
                if self.show_help: self.show_help=False; continue
                if self.btns.get("reset",pygame.Rect(0,0,0,0)).collidepoint(e.pos):
                    self.cut.clear(); self.show_min=False; self.won=False
                elif self.btns.get("min",pygame.Rect(0,0,0,0)).collidepoint(e.pos):
                    self.show_min=not self.show_min
                else:
                    edge=self._edge_at(e.pos)
                    if edge:
                        if edge in self.cut: self.cut.remove(edge)
                        else: self.cut.add(edge); self.attempts+=1
                        self.show_min=False
                        if self._valid() and self._cut_cost()==self.mf: self.won=True
            if e.type==pygame.KEYDOWN:
                if e.key==pygame.K_ESCAPE: return False if not self.show_help else [self.__setattr__('show_help',False) or True][0]
                if e.key==pygame.K_r: self.cut.clear(); self.show_min=False; self.won=False
                if e.key==pygame.K_m: self.show_min=not self.show_min
                if e.key==pygame.K_h: self.show_help=not self.show_help
                if e.key in(pygame.K_f,pygame.K_F11):
                    self.full=not self.full
                    self.screen=pygame.display.set_mode((0,0),pygame.FULLSCREEN) if self.full else pygame.display.set_mode((W0,H0),pygame.RESIZABLE)
                    self.gw=int(self.screen.get_width()*.74)
        return True

    def run(self):
        while self._events():
            self.tick+=1
            self.screen.fill(C["bg"])
            self._draw_graph(); self._draw_panel()
            pygame.display.flip(); self.clock.tick(60)
        pygame.quit()

if __name__=="__main__":
    mf=compute_max_flow(); opt=compute_min_cut()
    print(f"\n{'='*60}")
    print(f"  PIPELINE DEFENDER — Informacion fillestar")
    print(f"{'='*60}")
    print(f"  Graf: {len(set(n for u,v,_ in EDGES for n in (u,v)))} nyje, {len(EDGES)} linja")
    print(f"  Max-Flow = {mf} MW")
    print(f"  Min-Cut optimal: {[(NAMES[u],NAMES[v]) for u,v in opt]}")
    print(f"  Kosto Min-Cut = {sum(c for u,v,c in EDGES if (u,v) in opt)} MW")
    print(f"{'='*60}\n")
    Game().run()


pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html

  PIPELINE DEFENDER — Informacion fillestar
  Graf: 8 nyje, 16 linja
  Max-Flow = 37 MW
  Min-Cut optimal: [('Centrali', 'NS-A'), ('Centrali', 'NS-B'), ('Centrali', 'NS-C')]
  Kosto Min-Cut = 37 MW



---

## Detyra Shtëpie

> Të treja detyrat përdorin funksionet dhe strukturat e krijuara gjatë orës. Ndryshimet janë **minimale dhe të shënuara me koment `# TODO`**. Lexo me kujdes çdo koment para se të fillosh.

---

### Detyrë 1 — Max-Flow & Min-Cut: Rrjeti i Internetit

**Konteksti:** Një ISP (ofrues interneti) ka rrjetin e mëposhtëm. Gjen rrjedhën maksimale të të dhënave nga serveri kryesor (S) te përdoruesit (T), dhe identifiko prerjen minimale — lidhjet kritike që nëse dështojnë, e ndajnë rrjetin.

**Ndryshimet e kërkuara:**  
Kodi është identik me Ushtrimin 1 (Hidranti/Depozita). Duhet të ndryshosh **vetëm** vlerat e shënuara me `# TODO`.

```python
import pygame, math
from collections import defaultdict, deque
# [kopjo të gjithë kodin e Ushtrimit 1 këtu — funksionet mbeten IDENTIKE]
# Ndryshimet e vetme janë më poshtë:

pygame.init()
W0, H0 = 1440, 860
SOURCE, SINK = 0, 5

# TODO 1: Zëvendëso EDGES me rrjetin e internetit
# Rrjeti aktual (hidranti): [(0,1,10),(0,2,8),(1,3,6),(1,4,4),(2,4,7),(3,5,9),(4,5,8)]
EDGES = [
    (0, 1, 20),   # S    → Router-A    (fiber optike, 20 Gbps)
    (0, 2, 15),   # S    → Router-B    (fiber optike, 15 Gbps)
    (1, 3, 12),   # Router-A → Switch-1 (10GbE)
    (1, 4,  8),   # Router-A → Switch-2 (10GbE)
    (2, 3,  7),   # Router-B → Switch-1 (10GbE)
    (2, 4, 10),   # Router-B → Switch-2 (10GbE)
    (3, 5, 14),   # Switch-1 → T        (dalja veriore)
    (4, 5, 11),   # Switch-2 → T        (dalja jugore)
]

# TODO 2: Zëvendëso NAMES me emrat e rrjetit
NAMES = {
    0: "Serveri",
    1: "Router-A", 2: "Router-B",
    3: "Switch-1", 4: "Switch-2",
    5: "Klientët"
}

# [të gjitha funksionet dhe klasa App mbeten IDENTIKE me Ushtrimin 1]
# [thirrja App().run() mbetet e njëjtë]
```

**Pyetjet:**
1. Cila është rrjedha maksimale e të dhënave (Gbps) nga Serveri te Klientët?
2. Cilat janë lidhjet e Min-Cut? Çfarë ndodh nëse njëra prej tyre dështon?
3. Nëse ISP dëshiron të rrisë kapacitetin, cilin lidhje duhet të përmirësojë?

---

### Detyrë 2 — Ford-Fulkerson (DFS): Rrjeti Hekurudhor

**Konteksti:** Trena transportojnë mallra nga **Depo** (burim) te **Porti** (destinacion) nëpër stacione ndërmjetëse. Ford-Fulkerson me DFS gjen rrugët rritëse — por DFS mund të zgjedhë rrugë të gjata. Shiko sa iterime bën!

**Ndryshimet e kërkuara:**  
Kodi është identik me Ushtrimin 2 (Ford-Fulkerson/Edmonds-Karp Plotly). Duhet të ndryshosh **vetëm** vlerat e shënuara me `# TODO`. Ekzekuto **vetëm** Ford-Fulkerson (DFS).

```python
from collections import deque
import plotly.graph_objects as go
from IPython.display import Javascript, display
from ipywidgets import Button, HBox, HTML, Output, VBox
# [kopjo të gjithë kodin e Ushtrimit 2 këtu — klasa RrjedhaMaksimale mbetet IDENTIKE]

# TODO 1: Zëvendëso NYJET
NYJET = {
    0: ("S", "Depo"),
    1: ("A", "Stacioni-A"),
    2: ("B", "Stacioni-B"),
    3: ("C", "Stacioni-C"),
    4: ("D", "Stacioni-D"),
    5: ("T", "Porti"),
}

# TODO 2: Zëvendëso BRINJET me rrjetin hekurudhor
BRINJET = [
    (0, 1, 8),   # Depo → Stacioni-A  (8 trena/orë)
    (0, 2, 6),   # Depo → Stacioni-B  (6 trena/orë)
    (1, 3, 5),   # Stacioni-A → C
    (1, 4, 4),   # Stacioni-A → D
    (2, 3, 3),   # Stacioni-B → C
    (2, 4, 7),   # Stacioni-B → D
    (3, 5, 6),   # Stacioni-C → Porti
    (4, 5, 5),   # Stacioni-D → Porti
]

# TODO 3: Zëvendëso POZICIONET (koordinata x,y për Plotly)
POZICIONET = {
    0: (0, 0), 1: (2, 1), 2: (2,-1),
    3: (4, 1), 4: (4,-1), 5: (6, 0),
}

# TODO 4: Zëvendëso RENDI_DFS me rendin e eksplorimit për DFS
# Rendit fqinjët e çdo nyje — DFS i ndjek në këtë rend
RENDI_DFS = {
    0: [1, 2], 1: [3, 4], 2: [3, 4],
    3: [5],    4: [5],    5: [],
}

BURIMI, KULMI = 0, 5

# TODO 5: Krijo vetëm Ford-Fulkerson (DFS), jo Edmonds-Karp
ford_fulkerson = RrjedhaMaksimale(len(NYJET), BRINJET, BURIMI, KULMI, "dfs")
# edmonds_karp = ...  ← mos e krijo, nuk nevojitet

# TODO 6: Fshi butonin Edmonds-Karp — lër vetëm Ford-Fulkerson
# [pjesa e mbetur e UI mbetet IDENTIKE]
```

**Pyetjet:**
1. Sa iterime bën Ford-Fulkerson (DFS) për të gjetur max-flow?
2. Cilat rrugë zgjedh DFS? A janë rrugët më të shkurtra?
3. Nëse ndryshon RENDI_DFS, a ndryshon max-flow finale? Pse?

---

### Detyrë 3 — Edmonds-Karp (BFS): Krahasimi me Ford-Fulkerson

**Konteksti:** Përdor **të njëjtin rrjet hekurudhor** nga Detyra 2, por tani apliko **Edmonds-Karp (BFS)**. Krahaso numrin e iterimeve dhe rrugët e zgjedhura.

**Ndryshimet e kërkuara:**  
Kodi është identik me Detyrën 2. Ndryshimi i **vetëm** është strategjia e kërkimit.

```python
# [kopjo të gjithë kodin e Detyrës 2]

# TODO 1: Krijo Edmonds-Karp (BFS) — ndryshimi i vetëm është "bfs"
ford_fulkerson = RrjedhaMaksimale(len(NYJET), BRINJET, BURIMI, KULMI, "dfs")
edmonds_karp   = RrjedhaMaksimale(len(NYJET), BRINJET, BURIMI, KULMI, "bfs")
# ↑ Kjo është e GJITHË ndryshimi algoritmik — BFS vs DFS!

# TODO 2: Aktivizo të dy butonat (si në Ushtrimin 2 origjinal)
# [UI me të dy butonat — FF dhe EK — mbetet identik me Ushtrimin 2]

# TODO 3 (pyetje analitike — jo kod): plotëso tabelën:
# ┌─────────────────┬────────────────┬─────────────────┐
# │                 │ Ford-Fulkerson │  Edmonds-Karp   │
# ├─────────────────┼────────────────┼─────────────────┤
# │ Strategjia      │      DFS       │       BFS       │
# │ Max-Flow finale │     ???        │      ???        │
# │ Nr. iterimesh   │     ???        │      ???        │
# │ Rruga e 1-rë    │     ???        │      ???        │
# │ Kompleksiteti   │   O(E · f*)    │   O(V · E²)     │
# └─────────────────┴────────────────┴─────────────────┘
```

**Pyetjet:**
1. A japin të njëjtin max-flow final? Pse duhet të jetë kështu?
2. Cili algoritëm bën më pak iterime? Pse BFS garanton rrugën më të shkurtër?
3. **Thelbi i orës:** Nëse max-flow = 9 trena/orë, cila është kosto e min-cut? Si e verifikon?
4. Ford-Fulkerson = Edmonds-Karp, ndryshimi i vetëm është \_\_\_\_\_\_. (plotëso me një fjalë)
